# Cisco Skill Scanner on GitSkills — RQ1 experiment notebook

**MSR 2027 Mining Challenge · SkillGuard team · RQ1 (how skill-package context affects security findings)**

This notebook takes agent skills from the GitSkills SQLite database, rebuilds each skill on disk in two versions
(the `SKILL.md` instruction file alone, and the full skill folder with its supporting files), runs the Cisco AI
Defense Skill Scanner over both versions with two analyzer settings, and turns the scanner's JSON reports into
tables, charts and a written `results_summary.md`.

| Question | What we compare |
|---|---|
| **RQ1-A Context and detection** | the same skill scanned as `SKILL.md` only vs. as its full folder |
| **RQ1-B Security behavior coverage** | which Cisco rules and categories fire, and how they map to our research categories |
| **RQ1-C Limitations and relevance** | what Cisco does not report, how much the analyzer configuration matters, and how reliable each scan was |

### Vocabulary we keep separate

A scanner finding is **evidence, not a verdict**. We distinguish:

1. an **observed operation** (the code really does send an HTTP request),
2. a **scanner warning** (Cisco matched a rule on that operation),
3. a **potential vulnerability** (a human judged that the operation could be abused), and
4. **confirmed malicious behavior** (a human confirmed intent).

The notebook only produces items 1 and 2. Items 3 and 4 need a person; Section 14 writes a manual-review file
for that.

### Two ways to run it

| `RUN_MODE` | what happens | is Cisco called? |
|---|---|---|
| `"scan"` | select skills from the database, export them, scan every configuration, analyse | yes |
| `"reanalyze"` | copy a saved run (exported files and raw Cisco reports) into a new run folder and analyse it again with the current notebook code | **no** |

Every execution writes to a **new run folder**, `results/cisco_dataset_runs/<database>/runs/<run ID>/`. An earlier
run is never overwritten, and a reanalysis only *reads* the run it analyses.

### Safety

* The database is opened **read-only**. Nothing here writes to it.
* Skill contents are **untrusted**. The notebook writes them to disk and hands the folder to the scanner; it
  **never executes** a skill's scripts, and the scanner does not execute them either (it uses pattern, YARA,
  AST-dataflow and bytecode analysis). Reanalysis only reads saved files.
* Supporting-file paths from the database are checked before writing so nothing can escape the export folder.
* Run folders contain exported untrusted files; `results/cisco_dataset_runs/` is git-ignored. Do not commit it.

### How to run it

1. Select the project `.venv` kernel in VS Code.
2. Change the values in **Section 1 (Settings)**. Nothing else needs editing for a normal run.
3. **Run All**. Every section depends only on the sections above it. Re-running the whole notebook creates a new
   run folder; re-running a single cell in the middle is not supported.

Section map: **1** settings · **2** imports · **3** database · **4** scanner configurations · **5** select skills ·
**6** create the run folder · **7** export or copy the skills · **8** run Cisco / load reports and classify every
scan · **9** findings table and evidence check · **10** save tables · **11** A security findings ·
**12** B context comparison · **13** C analyzer comparison · **14** D reliability and E manual review ·
**15** results summary and figures · **16** interpretation and scaling ·
**17** researcher report (`START_HERE.md` + dashboard).

## 1 — Settings

Everything a researcher normally changes is in the next cell. Each setting has a comment with example values.

* **Run mode.** `"scan"` runs Cisco. `"reanalyze"` re-reads a saved run named by `SOURCE_RUN` (a folder that
  contains one sub-folder per skill with `manifest.json` and `reports/`). Both the original pilot layout
  (`pilot_canary_backup`, `agent_skills_sample`) and the new run folders work as a source.
* **Run name.** `EXPERIMENT_NAME` is a short label such as `pilot-x` or `batch-01`. The run ID becomes
  `<date>_<time>_<EXPERIMENT_NAME>`, for example `2026-11-03_151200_pilot-x`, so pilots and batches are easy to
  tell apart and two runs never share a folder. Setting `RUN_ID` yourself is allowed, but the notebook refuses to
  reuse a folder that already exists.
* **Database.** `"sample"` uses the official 13k-skill sample. For the full release give a path relative to the
  project root (for example `"data/full/agent_skills.db"`) or an absolute path. Reanalysis does not need the
  database; it is used only to record the dataset hash when it is present.
* **Selection (scan mode only).** `pilot` scans one known pilot skill; `exact` scans one repo/path you type;
  `search` only *previews* skills whose `SKILL.md` contains a keyword (no scanning); `batch` scans the first
  `MAX_SKILLS` keyword matches. Keyword search is a convenience for exploring, not our sampling method.
  `manifest` scans every skill listed in a saved sample manifest (`SAMPLE_MANIFEST`, written by
  `03_stratified_sample.ipynb`); it has no skill limit, checks that the database and every `SKILL.md` still match
  what was sampled, and records each skill's sampling stratum in all exports.
* **Scanner (scan mode only).** `POLICY` is Cisco's scan-policy preset; `balanced` is Cisco's default and is what
  the first pilot used. Keep it fixed within a study.
* **Optional analyzers** are off by default. `ENABLE_OSV` queries the OSV.dev vulnerability database over the
  network (no key). `ENABLE_LLM_CONFIGS` adds two LLM-judge configurations; it needs an API key in
  `SKILL_SCANNER_LLM_API_KEY`, costs money per scan, and **sends skill contents to the model provider**. Do not
  enable it without team agreement.

In [ ]:
# ---------------------------------------------------------------- Run mode --
RUN_MODE = "reanalyze"            # "scan" = select, export and scan skills with Cisco | "reanalyze" = analyse a saved run again (no scanning)
SOURCE_RUN = "agent_skills_sample/runs/2026-11-03_173516_sample30-seed2027"  # reanalyze only: saved run folder, relative to OUTPUT_ROOT or absolute, e.g. "agent_skills_sample/runs/2026-11-03_151200_pilot-x"
EXPERIMENT_NAME = "sample30-seed2027-reanalysis"  # short label at the end of the run ID, e.g. "pilot-x", "pilot-loa-credentials", "batch-01"
RUN_ID = None                     # None = "<date>_<time>_<EXPERIMENT_NAME>" (recommended); a fixed ID must not exist yet
OUTPUT_ROOT = "results/cisco_dataset_runs"  # run folders are created in OUTPUT_ROOT/<database>/runs/<run ID>/

# ---------------------------------------------------------------- Database --
DATABASE = "sample"               # "sample"  or a path such as "data/full/agent_skills.db"
HASH_DATABASE = True              # record the SHA-256 of the database file (~1 s for the sample; set False for a 44 GB file)

# ------------------------------------------------- Skill selection (scan mode) --
SELECTION_MODE = "manifest"          # "pilot" | "exact" | "search" | "batch" | "manifest"
PILOT = "canary-tokens"           # one of the PILOTS keys below
EXACT_REPO = "AceDataCloud/Skills"   # used when SELECTION_MODE == "exact"
EXACT_PATH = "skills/x/SKILL.md"
SEARCH_KEYWORD = "API_KEY"        # used by "search" and "batch"; case-insensitive text in SKILL.md
PREVIEW_LIMIT = 15                # rows shown in search mode (1..100)
MAX_SKILLS = 3                    # skills scanned in batch mode (1..20); larger studies run as several documented batches
SAMPLE_MANIFEST = "results/samples/agent_skills_sample_seed2027_scripts10_supporting_no_scripts10_skill_md_only10/sample_manifest.csv"  # used by "manifest" (no skill limit)

PILOTS = {
    # label: (repository, path of SKILL.md inside the repository)
    "loa-credentials": ("0xHoneyJar/loa-dixie", ".claude/skills/managing-credentials/SKILL.md"),
    "canary-tokens": ("26zl/cybersec-toolkit", ".claude/skills/implementing-canary-tokens-for-network-intrusion/SKILL.md"),
    "x": ("AceDataCloud/Skills", "skills/x/SKILL.md"),
}

# ---------------------------------------------------------- Scanner (scan mode) --
POLICY = "balanced"               # Cisco preset: "balanced" (default) | "strict" | "permissive" | "low-noise" | "quiet"
LENIENT = False                   # True adds --lenient (tolerate malformed SKILL.md frontmatter); keep False for comparability
SCAN_TIMEOUT_SECONDS = 300        # one scanner call normally takes 5-10 s; this only stops a hung process
ALLOW_INCOMPLETE_FOLDERS = False  # True scans "full" folders even when GitSkills did not store every file (reported as preliminary)

# ------------------------------------------------------------ Optional analyzers --
ENABLE_OSV = False                # add --use-osv to every configuration (dependency vulnerabilities; network, no key)
ENABLE_LLM_CONFIGS = False        # add two "LLM judge" configurations; needs SKILL_SCANNER_LLM_API_KEY, paid, sends content to a provider
LLM_PROVIDER = "anthropic"        # "anthropic" | "openai" | "openai-compatible" (only used when ENABLE_LLM_CONFIGS is True)

# Basic sanity checks so a typo fails here, not ten sections later.
assert RUN_MODE in ("scan", "reanalyze"), f"Unknown RUN_MODE: {RUN_MODE!r}"
assert SELECTION_MODE in ("pilot", "exact", "search", "batch", "manifest"), f"Unknown SELECTION_MODE: {SELECTION_MODE!r}"
assert PILOT in PILOTS, f"Unknown PILOT: {PILOT!r}; choose from {sorted(PILOTS)}"
assert 1 <= PREVIEW_LIMIT <= 100
assert 1 <= MAX_SKILLS <= 20, "Batch cap is 20 skills per run; split larger studies into documented batches"
assert POLICY in ("balanced", "strict", "permissive", "low-noise", "quiet"), f"Unknown POLICY: {POLICY!r}"
assert EXPERIMENT_NAME and all(ch.isalnum() or ch in "-_" for ch in EXPERIMENT_NAME), \
    "EXPERIMENT_NAME may only use letters, digits, '-' and '_'"
print("Settings loaded. RUN_MODE =", RUN_MODE, "| EXPERIMENT_NAME =", EXPERIMENT_NAME,
      "| SOURCE_RUN =", SOURCE_RUN if RUN_MODE == "reanalyze" else "(not used)")

## 2 — Imports and project folders

Standard-library modules we rely on:

* `sqlite3` talks to the GitSkills database; `pandas` turns query results into DataFrames (tables).
* `pathlib.Path` represents file paths. `ROOT / "data" / "sample"` builds a path piece by piece, and
  `.resolve()` turns it into an absolute path with symlinks and `..` removed.
* `hashlib` computes SHA-256 hashes so we can prove which bytes were scanned and that copies are identical.
* `subprocess` launches the Cisco command-line tool as a separate process (Section 8).
* `shutil.copytree` copies a saved run's skill folders in reanalysis mode (Section 7).
* `json` reads Cisco's reports; `time` measures how long each scan takes.

The project root is found by walking up from the notebook's folder until a folder with both `notebooks/` and
`data/` is found, so the notebook works whether VS Code starts it in `notebooks/` or in the repository root.

In [ ]:
import hashlib
import json
import os
import platform
import re
import shutil
import sqlite3
import subprocess
import sys
import tempfile
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", 80)   # characters shown per table cell; raise to read longer text
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

cwd = Path.cwd().resolve()
ROOT = next((folder for folder in (cwd, *cwd.parents) if (folder / "notebooks").is_dir() and (folder / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project repository (it needs the notebooks/ and data/ folders).")

OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT).expanduser()
if not OUTPUT_ROOT_PATH.is_absolute():
    OUTPUT_ROOT_PATH = ROOT / OUTPUT_ROOT_PATH
OUTPUT_ROOT_PATH = OUTPUT_ROOT_PATH.resolve()


def now_utc():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def sha256_of_file(path, chunk_bytes=8 * 1024 * 1024):
    """SHA-256 of a file, read in chunks so very large files do not need to fit in memory."""
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


def show_path(path):
    """Path relative to the project root when possible (shorter, and the same on every machine)."""
    path = Path(path)
    try:
        return path.resolve().relative_to(ROOT).as_posix()
    except ValueError:
        return str(path)


print("Project root :", ROOT)
print("Output root  :", show_path(OUTPUT_ROOT_PATH))

## 3 — Connect to the GitSkills database (read-only)

GitSkills stores every `SKILL.md` found on GitHub in the `artifacts` table and the other files of each skill
folder in `artifact_siblings`. Two schema facts matter for this experiment:

* **Copies share text.** The same `SKILL.md` is often vendored into many repositories. Copies share a
  `file_sha`; only one *representative* row per `file_sha` (`dedup_primary = 1`) stores the text and the
  supporting files. If you ask for a copy, Section 5 resolves it to its representative.
* **Not every supporting file has text.** `artifact_siblings.content_fetched` is `1` when the file text is
  stored and `2` when it was skipped (binary, oversized, or the folder had too many files). A folder with any
  `content_fetched != 1` file cannot be rebuilt completely; Section 7 records that and Section 8 skips the
  full-folder scan unless `ALLOW_INCOMPLETE_FOLDERS` is set.

The connection uses SQLite's `mode=ro` URI flag plus `PRAGMA query_only`, so an accidental `UPDATE` would fail
instead of changing the dataset. `sql("SELECT ...", (params,))` is the only way the notebook reads the
database; `?` placeholders keep values out of the SQL text.

In reanalysis mode the database is optional: if the file is missing, the notebook continues without it.

**Output:** connection details, the database hash (for the reproducibility record), and row counts.

In [ ]:
if DATABASE == "sample":
    DB_PATH = ROOT / "data" / "sample" / "agent_skills_sample.db"
else:
    DB_PATH = Path(DATABASE).expanduser()
    if not DB_PATH.is_absolute():
        DB_PATH = ROOT / DB_PATH
DB_PATH = DB_PATH.resolve()
DB_TAG = re.sub(r"[^a-zA-Z0-9_-]", "_", DB_PATH.stem)[:45]   # short safe name used in output folder names
DB_SHA256 = None
conn = None

if DB_PATH.is_file():
    # Read-only connection: the URI flag refuses writes at the file level, the PRAGMA refuses them at the SQL level.
    conn = sqlite3.connect(DB_PATH.as_uri() + "?mode=ro", uri=True)
    conn.execute("PRAGMA query_only = ON")
    conn.execute("PRAGMA busy_timeout = 15000")
    DB_SHA256 = sha256_of_file(DB_PATH) if HASH_DATABASE else None
elif RUN_MODE == "scan":
    local_dbs = sorted({*ROOT.glob("data/**/*.db"), *ROOT.glob("data/**/*.sqlite"), *ROOT.glob("data/**/*.sqlite3")})
    found = "\n".join(f"  - {show_path(p)} ({p.stat().st_size:,} bytes)" for p in local_dbs) or "  (none)"
    raise FileNotFoundError(f"Database not found: {DB_PATH}\nDatabases found under data/:\n{found}\nSet DATABASE in Section 1.")
else:
    print(f"Database not found at {show_path(DB_PATH)}. That is fine for reanalysis: no database access is needed.")


def sql(query, params=()):
    """Run a read-only SQL query and return the rows as a pandas DataFrame."""
    if conn is None:
        raise RuntimeError("No database connection (reanalysis without a database).")
    return pd.read_sql_query(query, conn, params=params)


def table_columns(table):
    """Return the set of column names of a table (PRAGMA table_info lists one row per column)."""
    return {row[1] for row in conn.execute(f'PRAGMA table_info("{table}")')}


if conn is not None:
    # The columns the rest of the notebook relies on. A database with a different schema needs an adapter first.
    REQUIRED_COLUMNS = {
        "artifacts": {"repo_full_name", "path", "filename", "content", "file_sha", "dedup_primary", "name",
                      "frontmatter_valid", "sibling_count", "has_scripts", "has_references",
                      "composition_fetched", "composition_truncated"},
        "artifact_siblings": {"repo_full_name", "artifact_path", "entry_name", "entry_type", "content", "content_fetched"},
    }
    for table, needed in REQUIRED_COLUMNS.items():
        missing = needed - table_columns(table)
        if missing:
            raise ValueError(f"Incompatible database schema: table {table!r} lacks columns {sorted(missing)}")

    print("Connected (read-only):", show_path(DB_PATH))
    print("Database tag         :", DB_TAG)
    print("Database SHA-256     :", DB_SHA256 or "(not computed)")
    print("Integrity quick_check:", conn.execute("PRAGMA quick_check").fetchone()[0])
    display(sql("""
        SELECT COUNT(*)                        AS skill_md_occurrences,
               SUM(dedup_primary)              AS distinct_skill_files,
               SUM(dedup_primary = 1 AND sibling_count > 0) AS distinct_with_supporting_files,
               COUNT(DISTINCT repo_full_name)  AS repositories
        FROM artifacts
    """))

## 4 — The Cisco scanner and our four configurations

[Cisco AI Defense Skill Scanner](https://github.com/cisco-ai-defense/skill-scanner) (PyPI package
`cisco-ai-skill-scanner`, command `skill-scanner`, pinned to 2.3.1 in `requirements.txt`) scans a **skill
folder** and reports *findings*. Each finding has a rule ID, a Cisco threat category, a severity (`CRITICAL`,
`HIGH`, `MEDIUM`, `LOW`, `INFO`), the analyzer that produced it, and where possible a file path, line number,
source snippet and remediation text. The overall verdict `is_safe` is simply "no `CRITICAL` or `HIGH` finding" —
a skill can be `is_safe = True` with several `MEDIUM` findings, so we never use the verdict alone.

| Analyzer | Enabled by | What it looks at |
|---|---|---|
| `static` | always | ~47 YAML signature rules (regexes per file type), 27 YARA rules, and Python checks on the manifest, hidden files, binaries, file-type mismatches, undeclared network use |
| `bytecode` | always | `.pyc` files that do not match their source |
| `pipeline` | always | shell command chains (`curl … \| sh`, taint through pipes) |
| `correlation` | always (policy) | bounded source-to-sink flows across files (e.g. sensitive read → network send) |
| `behavioral` | `--use-behavioral` | Python AST dataflow: env-var harvesting plus network, eval/exec with subprocess, cross-file flows; also code blocks inside Markdown (`MDBLOCK_*` rules) |
| `llm` | `--use-llm` | an LLM judge reads `SKILL.md` and scripts for semantic concerns; needs an API key; paid; content leaves the machine |
| `osv` | `--use-osv` | OSV.dev lookup of exactly pinned Python/JS dependencies; network, no key |

The **scan policy** (`--policy`) controls thresholds, allowlists, rule severity overrides and which core analyzers
run; `balanced` is Cisco's default. We pass it explicitly so every report records the same policy fingerprint.

Our experiment is a 2 × 2 design, built below as `SCAN_CONFIGS`:

| | default analyzers | `--use-behavioral` |
|---|---|---|
| **SKILL.md only** | configuration 1 | configuration 2 |
| **Full folder** | configuration 3 | configuration 4 |

Configurations 1 vs 3 and 2 vs 4 answer RQ1-A (context); 1 vs 2 and 3 vs 4 answer the analyzer part of RQ1-C.
Each configuration lists the analyzers that **must** appear in the report's `analyzers_used`; Section 8 uses that
list to detect scans where a requested analyzer silently did not run.

**Output:** the installed scanner version and a table of the configurations.

In [ ]:
scanner_candidates = [ROOT / ".venv/bin/skill-scanner", ROOT / ".venv/Scripts/skill-scanner.exe"]
SCANNER = next((str(p) for p in scanner_candidates if p.is_file()), None) or shutil.which("skill-scanner")

if SCANNER is None:
    if RUN_MODE == "scan":
        raise FileNotFoundError(
            "Cisco skill-scanner not found. Install the pinned version into the project environment with\n"
            "    .venv/bin/pip install -r requirements.txt\n"
            "or set RUN_MODE = 'reanalyze' to analyse saved reports without scanning."
        )
    INSTALLED_SCANNER_VERSION = "not installed"
else:
    version_proc = subprocess.run([SCANNER, "--version"], capture_output=True, text=True, timeout=60)
    INSTALLED_SCANNER_VERSION = (version_proc.stdout or version_proc.stderr).strip()
print("Scanner          :", SCANNER)
print("Installed version:", INSTALLED_SCANNER_VERSION,
      "(used for new scans)" if RUN_MODE == "scan" else "(not used: reanalysis reads saved reports)")

# Arguments shared by every configuration. JSON output is what Section 8 parses.
COMMON_ARGS = ["--format", "json", "--policy", POLICY]
if LENIENT:
    COMMON_ARGS.append("--lenient")
if ENABLE_OSV:
    COMMON_ARGS.append("--use-osv")

# Which exported folder each scope uses (Section 7 creates them).
SCOPE_FOLDER = {"SKILL.md only": "only", "Full folder": "full"}

SCAN_CONFIGS = [
    {"configuration": "SKILL.md only | default",    "scope": "SKILL.md only", "analyzer_mode": "default",
     "report_file": "skill_only_default.json",    "extra_args": [],                   "required_analyzers": ["static_analyzer"]},
    {"configuration": "SKILL.md only | behavioral", "scope": "SKILL.md only", "analyzer_mode": "behavioral",
     "report_file": "skill_only_behavioral.json", "extra_args": ["--use-behavioral"], "required_analyzers": ["static_analyzer", "behavioral_analyzer"]},
    {"configuration": "Full folder | default",      "scope": "Full folder",   "analyzer_mode": "default",
     "report_file": "full_folder_default.json",   "extra_args": [],                   "required_analyzers": ["static_analyzer"]},
    {"configuration": "Full folder | behavioral",   "scope": "Full folder",   "analyzer_mode": "behavioral",
     "report_file": "full_folder_behavioral.json","extra_args": ["--use-behavioral"], "required_analyzers": ["static_analyzer", "behavioral_analyzer"]},
]

if ENABLE_LLM_CONFIGS:
    if not os.environ.get("SKILL_SCANNER_LLM_API_KEY") and RUN_MODE == "scan":
        print("WARNING: ENABLE_LLM_CONFIGS is True but SKILL_SCANNER_LLM_API_KEY is not set; LLM configurations are NOT added.")
    else:
        llm_args = ["--use-llm", "--llm-provider", LLM_PROVIDER]
        SCAN_CONFIGS += [
            {"configuration": "SKILL.md only | llm", "scope": "SKILL.md only", "analyzer_mode": "llm",
             "report_file": "skill_only_llm.json", "extra_args": llm_args, "required_analyzers": ["static_analyzer", "llm_analyzer"]},
            {"configuration": "Full folder | llm",   "scope": "Full folder",   "analyzer_mode": "llm",
             "report_file": "full_folder_llm.json",  "extra_args": llm_args, "required_analyzers": ["static_analyzer", "llm_analyzer"]},
        ]
        if RUN_MODE == "scan":
            print("LLM configurations added. Skill contents WILL be sent to the configured provider and billed.")

CONFIG_ORDER = [cfg["configuration"] for cfg in SCAN_CONFIGS]
CONFIG_BY_NAME = {cfg["configuration"]: cfg for cfg in SCAN_CONFIGS}
display(pd.DataFrame(SCAN_CONFIGS).assign(common_args=" ".join(COMMON_ARGS)))

## 5 — Select the skills

This section produces `selected_df`, one row per skill.

**Scan mode.** `lookup_skill(repo, path)` joins each requested row to its **representative**
(`rep.file_sha = a.file_sha AND rep.dedup_primary = 1`) so copies resolve to the row that actually stores the text
and supporting files. It also pulls the GitSkills fields that matter for interpreting results later:
`frontmatter_valid`, `sibling_count`, `has_scripts`, `has_references`, `composition_fetched` and
`composition_truncated`. In `search` mode nothing is selected: it previews matches so you can pick one for
`exact` mode.

**Manifest mode.** The skills are the rows of a saved sample manifest, in manifest order. Before anything is
exported the notebook checks that the database file has the SHA-256 recorded in the manifest and that every sampled
skill is still its own representative with the same `file_sha`; otherwise it stops. Each skill keeps its
`sample_stratum`, which is written to `manifest.json`, `scan_summary.csv` and `individual_findings.csv`.

**Reanalysis mode.** The skills are whatever the saved run contains: every sub-folder of `SOURCE_RUN` that has a
`manifest.json`. The saved run's own `run_metadata.json`, if present, tells us which scanner version and policy
produced its reports; old reports do not record that themselves.

**Output:** the selection table (or the search preview).

In [ ]:
def lookup_skill(repo, path):
    """Describe one skill and resolve it to the representative row that stores its text and files."""
    rows = sql("""
        SELECT a.repo_full_name AS repo, a.path AS skill_path, a.filename, a.dedup_primary,
               rep.repo_full_name AS rep_repo, rep.path AS rep_path,
               rep.name, rep.frontmatter_valid, rep.sibling_count, rep.has_scripts, rep.has_references,
               rep.composition_fetched, rep.composition_truncated, rep.file_sha
        FROM artifacts a
        JOIN artifacts rep ON rep.file_sha = a.file_sha AND rep.dedup_primary = 1
        WHERE a.repo_full_name = ? AND a.path = ?
    """, (repo, path))
    if rows.empty:
        raise ValueError(f"Skill not found in the database: {repo} / {path}")
    return rows.iloc[0]


def search_skills(keyword, limit):
    """Distinct skills whose SKILL.md text contains `keyword` (case-insensitive). Preview columns only."""
    return sql("""
        SELECT repo_full_name AS repo, path AS skill_path, name, sibling_count, has_scripts, has_references,
               frontmatter_valid, substr(content, 1, 100) AS preview
        FROM artifacts
        WHERE dedup_primary = 1 AND instr(lower(content), lower(?)) > 0
        ORDER BY repo_full_name, path
        LIMIT ?
    """, (keyword, limit))

In [ ]:
SOURCE_DIR, SOURCE_METADATA, SOURCE_SKILL_DIRS = None, {}, []
SAMPLE_INFO = None            # filled in manifest mode

if RUN_MODE == "reanalyze":
    SOURCE_DIR = Path(SOURCE_RUN).expanduser()
    if not SOURCE_DIR.is_absolute():
        SOURCE_DIR = OUTPUT_ROOT_PATH / SOURCE_DIR
    SOURCE_DIR = SOURCE_DIR.resolve()
    SOURCE_SKILL_DIRS = sorted(p.parent for p in SOURCE_DIR.glob("*/manifest.json"))
    if not SOURCE_SKILL_DIRS:
        raise FileNotFoundError(f"No saved skills (sub-folders with manifest.json) in {SOURCE_DIR}. Check SOURCE_RUN, "
                                "or set RUN_MODE = 'scan'.")
    if (SOURCE_DIR / "run_metadata.json").is_file():
        SOURCE_METADATA = json.loads((SOURCE_DIR / "run_metadata.json").read_text(encoding="utf-8"))
    source_sample = SOURCE_DIR / "sample" / "sample_manifest.csv"
    if source_sample.is_file():      # a manifest-based run: keep its sample definition and its skill order
        sample_manifest = pd.read_csv(source_sample)
        position = {skill_id: i for i, skill_id in enumerate(sample_manifest["skill_id"])}
        SOURCE_SKILL_DIRS.sort(key=lambda d: (position.get(d.name, len(position)), d.name))
        SAMPLE_INFO = {"manifest": show_path(source_sample), "manifest_sha256": sha256_of_file(source_sample),
                       "sample_id": str(sample_manifest["sample_id"].iloc[0]), "skills": len(sample_manifest),
                       "database_sha256": str(sample_manifest["database_sha256"].iloc[0]),
                       "strata": sample_manifest["stratum"].value_counts(sort=False).to_dict()}
    rows = []
    for skill_dir in SOURCE_SKILL_DIRS:
        manifest = json.loads((skill_dir / "manifest.json").read_text(encoding="utf-8"))
        rows.append({"skill_id": skill_dir.name, "repo": manifest.get("repo"), "skill_path": manifest.get("skill_path")})
    selected_df = pd.DataFrame(rows)
    selections = list(zip(selected_df["repo"], selected_df["skill_path"]))
    print(f"Reanalysing {len(selected_df)} saved skill(s) from {show_path(SOURCE_DIR)}")
    print("Saved run metadata:", "found" if SOURCE_METADATA else "not found (scanner version and policy are read from the reports where possible)")
    display(selected_df)
else:
    if SELECTION_MODE == "pilot":
        selections = [PILOTS[PILOT]]
    elif SELECTION_MODE == "exact":
        if not EXACT_REPO.strip() or not EXACT_PATH.strip():
            raise ValueError("Set EXACT_REPO and EXACT_PATH in Section 1")
        selections = [(EXACT_REPO.strip(), EXACT_PATH.strip())]
    elif SELECTION_MODE == "search":
        matches = search_skills(SEARCH_KEYWORD, PREVIEW_LIMIT)
        print(f"Preview of up to {PREVIEW_LIMIT} distinct skills containing {SEARCH_KEYWORD!r} (nothing will be scanned):")
        display(matches)
        selections = []
    elif SELECTION_MODE == "manifest":
        manifest_path = Path(SAMPLE_MANIFEST).expanduser()
        manifest_path = (manifest_path if manifest_path.is_absolute() else ROOT / manifest_path).resolve()
        sample_manifest = pd.read_csv(manifest_path)
        SAMPLE_INFO = {"manifest": show_path(manifest_path), "manifest_sha256": sha256_of_file(manifest_path),
                       "sample_id": str(sample_manifest["sample_id"].iloc[0]), "skills": len(sample_manifest),
                       "database_sha256": str(sample_manifest["database_sha256"].iloc[0]),
                       "strata": sample_manifest["stratum"].value_counts(sort=False).to_dict()}
        if DB_SHA256 is None:
            print("WARNING: HASH_DATABASE is False, so the database snapshot cannot be compared with the sample.")
        elif DB_SHA256 != SAMPLE_INFO["database_sha256"]:
            raise ValueError("This database file is not the one the sample was drawn from "
                             f"(sample: {SAMPLE_INFO['database_sha256'][:16]}..., here: {DB_SHA256[:16]}...).")
        selections = list(zip(sample_manifest["repo"], sample_manifest["skill_path"]))
    else:  # batch
        matches = search_skills(SEARCH_KEYWORD, MAX_SKILLS)
        selections = list(matches[["repo", "skill_path"]].itertuples(index=False, name=None))

    selected_df = pd.DataFrame([lookup_skill(repo, path) for repo, path in selections])
    if SAMPLE_INFO is not None and not selected_df.empty:
        # Every sampled skill must still be its own representative with the same SKILL.md hash.
        same = (selected_df["file_sha"].values == sample_manifest["file_sha"].values) & (selected_df["dedup_primary"].values == 1)
        if len(selected_df) != len(sample_manifest) or not same.all():
            raise ValueError(f"{int((~same).sum())} sampled skill(s) no longer match the manifest (file_sha or representative changed).")
        selected_df["sample_id"] = sample_manifest["sample_id"].values
        selected_df["sample_stratum"] = sample_manifest["stratum"].values
        print(f"Manifest check passed: {len(selected_df)} skills from sample {SAMPLE_INFO['sample_id']}; database hash and "
              f"every SKILL.md hash match. Strata: {SAMPLE_INFO['strata']}")
    print(f"Selected {len(selected_df)} skill(s).")
    if not selected_df.empty:
        copies = selected_df[selected_df["dedup_primary"] != 1]
        if not copies.empty:
            print(f"{len(copies)} selection(s) are copies; their text is taken from the representative row (rep_repo / rep_path).")
        display(selected_df[[c for c in ["sample_stratum", "repo", "skill_path", "name", "frontmatter_valid", "sibling_count",
                                         "has_scripts", "has_references", "composition_fetched", "composition_truncated",
                                         "rep_repo", "rep_path"] if c in selected_df.columns]])
    elif SELECTION_MODE == "search":
        print("SEARCH MODE: preview only. Set SELECTION_MODE = 'exact' with a repo/path above, or 'batch', to scan.")

## 6 — Create this run's folder

Every execution gets its own folder, so a later pilot or batch can never overwrite an earlier one:

```
results/cisco_dataset_runs/<database>/runs/<run ID>/
    <skill_id>/only/SKILL.md            exported SKILL.md (scope "SKILL.md only")
    <skill_id>/full/...                 SKILL.md + supporting files (scope "Full folder")
    <skill_id>/manifest.json            what was exported, SHA-256 of every file, missing files
    <skill_id>/reports/*.json           raw Cisco reports, exactly as Cisco printed them
    <skill_id>/reports/*.run.json       run record per scan: command, exit code, timing, report hash
    scan_summary.csv                    one row per skill x configuration
    individual_findings.csv             one row per Cisco finding
    context_comparison.csv              SKILL.md only vs full folder, finding by finding
    analyzer_comparison.csv             default vs behavioral, finding by finding
    manual_review_template.csv          one row per distinct finding, blank reviewer columns
    run_metadata.json                   settings, versions, hashes, outcome counts
    results_summary.md                  the written summary (Section 15)
    figures/*.png                       exported charts
```

The run ID is `<date>_<time>_<EXPERIMENT_NAME>` unless `RUN_ID` is set. `mkdir(exist_ok=False)` makes Python
raise an error instead of reusing a folder that already exists. The original pilot folders
(`agent_skills_sample/`, `pilot_canary_backup/`) are left exactly as they are and can be used as `SOURCE_RUN`.

When nothing was selected (search mode), no folder is created and the later sections only print notices.

In [ ]:
RUN_DIR = None
RUN_ID_USED = None
RUN_STARTED_AT = now_utc()

if selected_df.empty:
    print("No skills selected: no run folder is created and nothing will be saved.")
else:
    run_id = RUN_ID or f"{datetime.now():%Y-%m-%d_%H%M%S}_{EXPERIMENT_NAME}"
    new_run_dir = OUTPUT_ROOT_PATH / DB_TAG / "runs" / run_id
    if SOURCE_DIR is not None and new_run_dir.resolve() == SOURCE_DIR:
        raise ValueError("The new run folder would be the same as SOURCE_RUN; choose a different RUN_ID.")
    try:
        new_run_dir.mkdir(parents=True, exist_ok=False)   # never reuse an existing experiment folder
    except FileExistsError:
        raise FileExistsError(f"Run folder already exists: {show_path(new_run_dir)}\n"
                              "Set RUN_ID = None (a new timestamped ID) or choose a different RUN_ID.") from None
    # Only now, after the folder was created by THIS run, do later sections get a place to write.
    # If the cell above fails, RUN_DIR stays None and nothing later can write into someone else's run.
    RUN_DIR, RUN_ID_USED = new_run_dir, run_id
    (RUN_DIR / "figures").mkdir()
    if SAMPLE_INFO is not None:      # keep the exact sample definition next to the results
        (RUN_DIR / "sample").mkdir()
        sample_dir = (ROOT / SAMPLE_INFO["manifest"]).parent
        for name in ("sample_manifest.csv", "sampling_metadata.json"):
            if (sample_dir / name).is_file():
                shutil.copy2(sample_dir / name, RUN_DIR / "sample" / name)
    print("Run ID     :", RUN_ID_USED)
    print("Run folder :", show_path(RUN_DIR))

## 7 — Export the skills (scan mode) or copy the saved ones (reanalysis)

**Scan mode** writes each selected skill from the database into the run folder. Each step is a small function:

* **`skill_id_for`** builds a folder name from the skill's folder name plus a 12-character hash of
  `repo + path`, so two skills with the same folder name never collide and the name is stable across runs.
* **`safe_target_path`** rejects supporting-file names that could escape the folder (`..`, absolute paths,
  backslashes) or overwrite the instruction file. A *nested* `sub/SKILL.md` is allowed and recorded.
* **`write_exact_bytes`** writes the UTF-8 bytes of the stored text and refuses to overwrite a file whose
  bytes differ. Comparing bytes (not decoded text) matters because some skills use Windows line endings.
* **`export_skill`** writes both scopes, asserts the two `SKILL.md` files are byte-identical (the only intended
  difference between scopes is the presence of supporting files), and decides **folder completeness**:
  complete means GitSkills fetched the folder listing without truncation **and** stored text for every file.

**Reanalysis mode** copies each saved skill folder (exported files, manifest, raw reports) into the new run folder
with `shutil.copytree`, which only reads the source. It then re-checks the copy: every file listed in the
manifest must exist with the recorded SHA-256, and `only/SKILL.md` must equal `full/SKILL.md` byte for byte.
The result is in the `integrity` column.

**Output:** `exports` (one record per skill) and `exports_df`, a table of what was written or copied.

In [ ]:
def skill_id_for(repo, skill_path):
    """Stable, filesystem-safe folder name: <skill folder name>-<12 hex chars of sha256(repo + path)>."""
    folder_name = Path(skill_path).parent.name
    stem = re.sub(r"[^a-zA-Z0-9_-]", "-", folder_name)[:35].strip("-") or "skill"
    digest = hashlib.sha256(f"{repo}\n{skill_path}".encode()).hexdigest()[:12]
    return f"{stem}-{digest}"


def safe_target_path(folder, entry_name):
    """Where a supporting file goes inside `folder`, or raise if the name is unsafe.

    GitSkills sibling names are relative to the skill folder, e.g. "scripts/agent.py".
    """
    rel = Path(entry_name)
    if not entry_name or rel.is_absolute() or ".." in rel.parts or "\\" in entry_name:
        raise ValueError(f"Unsafe supporting-file path: {entry_name!r}")
    if len(rel.parts) == 1 and rel.name.lower() == "skill.md":
        # A top-level file named SKILL.md (any letter case; macOS file systems ignore case) would overwrite the instruction file.
        raise ValueError(f"Supporting file would overwrite SKILL.md: {entry_name!r}")
    target = folder / rel
    if not target.resolve().is_relative_to(folder.resolve()):
        raise ValueError(f"Path escapes the export folder: {entry_name!r}")
    return target


def write_exact_bytes(path, data):
    """Write `data` (bytes) to `path`; if the file exists with different bytes, refuse instead of overwriting."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != data:
            raise ValueError(f"File differs from an earlier export; refusing to overwrite: {path}")
    else:
        path.write_bytes(data)


def export_skill(skill):
    """Write the two scope folders for one selected skill and return its manifest (also saved as manifest.json)."""
    skill_id = skill_id_for(skill.repo, skill.skill_path)
    skill_dir = RUN_DIR / skill_id
    folder_only, folder_full = skill_dir / "only", skill_dir / "full"

    text = sql("SELECT content FROM artifacts WHERE repo_full_name = ? AND path = ?", (skill.rep_repo, skill.rep_path))
    content = text.iloc[0]["content"] if not text.empty else None
    if not isinstance(content, str) or not content.strip():
        raise ValueError("SKILL.md text is not stored for this skill (or is empty)")
    skill_bytes = content.encode("utf-8")
    write_exact_bytes(folder_only / "SKILL.md", skill_bytes)
    write_exact_bytes(folder_full / "SKILL.md", skill_bytes)

    siblings = sql("""
        SELECT entry_name, entry_type, content, content_fetched
        FROM artifact_siblings
        WHERE repo_full_name = ? AND artifact_path = ?
        ORDER BY entry_name
    """, (skill.rep_repo, skill.rep_path))

    written, missing, nested_skill_files = [], [], []
    for item in siblings.itertuples(index=False):
        if item.entry_type != "file":
            continue                                   # directories are created implicitly
        target = safe_target_path(folder_full, item.entry_name)   # validate every name, even ones we cannot write
        if item.content_fetched != 1 or not isinstance(item.content, str):
            missing.append(item.entry_name)            # binary / oversized / capped folder: text not in GitSkills
            continue
        if Path(item.entry_name).name.lower() == "skill.md":
            nested_skill_files.append(item.entry_name)
        write_exact_bytes(target, item.content.encode("utf-8"))
        written.append(item.entry_name)

    if (folder_only / "SKILL.md").read_bytes() != (folder_full / "SKILL.md").read_bytes():
        raise AssertionError("SKILL.md bytes differ between scopes")   # should be impossible; checked anyway

    listing_complete = bool(pd.notna(skill.composition_fetched) and int(skill.composition_fetched) == 1
                            and pd.notna(skill.composition_truncated) and int(skill.composition_truncated) == 0)
    manifest = {
        "skill_id": skill_id,
        "repo": skill.repo, "skill_path": skill.skill_path, "original_filename": skill.filename,
        "representative_repo": skill.rep_repo, "representative_path": skill.rep_path, "file_sha": skill.file_sha,
        "database": str(DB_PATH), "database_sha256": DB_SHA256,
        "frontmatter_valid": int(skill.frontmatter_valid) if pd.notna(skill.frontmatter_valid) else None,
        "has_scripts": int(skill.has_scripts) if pd.notna(skill.has_scripts) else None,
        "has_references": int(skill.has_references) if pd.notna(skill.has_references) else None,
        "sibling_count_in_db": int(skill.sibling_count) if pd.notna(skill.sibling_count) else None,
        "listing_complete": listing_complete,
        "supporting_files_written": written,
        "missing_files": missing,
        "nested_skill_files": nested_skill_files,
        "folder_complete": listing_complete and not missing,
        "skill_md_sha256": hashlib.sha256(skill_bytes).hexdigest(),
        "files": [{"path": p.relative_to(folder_full).as_posix(), "bytes": p.stat().st_size, "sha256": sha256_of_file(p)}
                  for p in sorted(folder_full.rglob("*")) if p.is_file()],
        "exported_at": now_utc(),
        "sample_id": getattr(skill, "sample_id", None), "sample_stratum": getattr(skill, "sample_stratum", None),
    }
    (skill_dir / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    return manifest

In [ ]:
def copy_saved_skill(source_skill_dir):
    """Copy one saved skill folder into this run (the source is only read) and verify the copy against its manifest."""
    target = RUN_DIR / source_skill_dir.name
    shutil.copytree(source_skill_dir, target, ignore=shutil.ignore_patterns(".DS_Store"))
    manifest = json.loads((target / "manifest.json").read_text(encoding="utf-8"))
    problems = []
    for entry in manifest.get("files", []):
        copied = target / "full" / entry["path"]
        if not copied.is_file():
            problems.append(f"missing {entry['path']}")
        elif entry.get("sha256") and sha256_of_file(copied) != entry["sha256"]:
            problems.append(f"hash differs: {entry['path']}")
    only_md, full_md = target / "only" / "SKILL.md", target / "full" / "SKILL.md"
    if not only_md.is_file() or not full_md.is_file():
        problems.append("only/SKILL.md or full/SKILL.md missing")
    elif only_md.read_bytes() != full_md.read_bytes():
        problems.append("SKILL.md differs between scopes")
    integrity = "ok: copy matches manifest hashes" if not problems else "PROBLEM: " + "; ".join(problems)
    return manifest, integrity


def export_record(manifest, origin, integrity):
    """One row describing an exported skill. Works for new manifests and for older saved ones (missing keys -> None)."""
    files = manifest.get("files") or []
    folder_complete = manifest.get("folder_complete", manifest.get("composition_complete"))
    return {
        "skill_id": manifest["skill_id"], "repo": manifest.get("repo"), "skill_path": manifest.get("skill_path"),
        "origin": origin, "integrity": integrity,
        "frontmatter_valid": manifest.get("frontmatter_valid"),
        "has_scripts": manifest.get("has_scripts"), "has_references": manifest.get("has_references"),
        "supporting_files_exported": len([f for f in files if f["path"] != "SKILL.md"]),
        "files_missing": len(manifest.get("missing_files") or []),
        "missing_files": manifest.get("missing_files") or [],
        "listing_complete": manifest.get("listing_complete"),
        "folder_complete": bool(folder_complete) if folder_complete is not None else None,
        "nested_skill_files": manifest.get("nested_skill_files") or [],
        "sample_stratum": manifest.get("sample_stratum"),
    }


exports, export_errors = [], []

if RUN_MODE == "scan" and RUN_DIR is not None:
    for number, skill in enumerate(selected_df.itertuples(index=False), start=1):
        print(f"[{number}/{len(selected_df)}] {skill.repo} / {skill.skill_path}")
        try:
            manifest = export_skill(skill)
        except Exception as exc:          # one bad skill must not stop the batch; it is recorded, not hidden
            print(f"    EXPORT ERROR: {type(exc).__name__}: {str(exc)[:200]}")
            export_errors.append({"skill_id": skill_id_for(skill.repo, skill.skill_path), "repo": skill.repo,
                                  "skill_path": skill.skill_path, "sample_stratum": getattr(skill, "sample_stratum", None),
                                  "error": f"{type(exc).__name__}: {exc}"})
            continue
        record = export_record(manifest, "exported from database", "ok: written in this run")
        exports.append(record)
        print(f"    wrote SKILL.md + {record['supporting_files_exported']} supporting file(s); folder complete: {record['folder_complete']}"
              + (f"; missing text for {record['missing_files']}" if record["missing_files"] else "")
              + (f"; nested skill files: {record['nested_skill_files']}" if record["nested_skill_files"] else ""))
elif RUN_DIR is not None:
    for number, source_skill_dir in enumerate(SOURCE_SKILL_DIRS, start=1):
        manifest, integrity = copy_saved_skill(source_skill_dir)
        exports.append(export_record(manifest, f"copied from {show_path(SOURCE_DIR)}", integrity))
        print(f"[{number}/{len(SOURCE_SKILL_DIRS)}] copied {source_skill_dir.name}: {integrity}")

EXPORT_COLUMNS = ["skill_id", "repo", "skill_path", "origin", "integrity", "frontmatter_valid", "has_scripts", "has_references",
                  "supporting_files_exported", "files_missing", "listing_complete", "folder_complete", "sample_stratum"]
exports_df = pd.DataFrame(exports, columns=EXPORT_COLUMNS + ["missing_files", "nested_skill_files"])[EXPORT_COLUMNS]
print(f"\n{len(exports)} skill(s) ready in the run folder" + (f"; {len(export_errors)} export error(s)" if export_errors else ""))
display(exports_df)

## 8 — Run Cisco (scan mode) and classify every scan

For each skill and configuration, scan mode runs one command such as

```
skill-scanner scan results/.../full --format json --policy balanced --use-behavioral
```

`subprocess.run([...], capture_output=True, text=True, timeout=...)` starts that command as a separate process and
gives us its exit code, its standard output (the JSON report) and its standard error (warnings). Cisco's output is
saved **exactly as printed** in `reports/<configuration>.json`, written to a temporary file and renamed so a crash
never leaves a half-written report. Next to it, `reports/<configuration>.run.json` records the command, exit code,
standard-error tail, wall time, scanner version and the SHA-256 of the saved report.

Then **one function, `load_and_assess`, classifies every scan from the files on disk**. Scan mode calls it right
after scanning; reanalysis calls it on the copied reports. New and saved reports are therefore judged by exactly
the same rules.

| status | outcome | meaning | used in primary tables and comparisons? |
|---|---|---|---|
| `COMPLETE` | complete | valid report, every requested analyzer ran, Cisco reported no analyzer failure | yes |
| `CONTRACT_ONLY` | contract_only | valid report, every requested analyzer ran, and the **only** problem Cisco recorded is the known `FindingContract` rule-registration error (code `UNKNOWN_BUNDLED_PYTHON_RULE`, see below) | no — findings are kept and used **only in the sensitivity analysis** of Sections 12–13 |
| `PARTIAL` | partial | valid report, **but** Cisco listed any other analyzer failure (including a `FindingContract` error with another code), its rule-contract check failed without a matching entry, a requested analyzer is missing from `analyzers_used`, the LLM analyzer reported `LLM_ANALYSIS_FAILED`, or the recorded exit code was not 0 | no — findings are kept, exported and shown separately as preliminary evidence |
| `FAILED_TIMEOUT`, `FAILED_SCANNER_ERROR`, `FAILED_INVALID_JSON`, `FAILED_INVALID_FORMAT`, `FAILED_INTEGRITY` | failed | no usable report (or the report no longer matches its recorded hash) | no |
| `SKIPPED_INCOMPLETE_FOLDER` | excluded_incomplete_folder | **design exclusion, not a scan:** GitSkills did not store every supporting file, so the full-folder scan was deliberately not run (Cisco was never invoked) | no — the pair is reported as `excluded: incomplete folder` |
| `MISSING_REPORT` | missing | reanalysis only: the saved run has no report or run record for this configuration | no |
| `EXPORT_ERROR` | failed | the skill could not be exported, so nothing was scanned | no |

None of these is the same as "zero findings". A `COMPLETE` scan with no findings is the only zero.

**Planned vs executed.** Every skill has one *planned configuration* per entry in `SCAN_CONFIGS`. A configuration
is an *executed scan* only when Cisco was actually invoked (outcome complete, contract_only, partial or failed;
column `scan_executed`). Incomplete-folder exclusions, missing reports and export errors are planned but not executed,
and the reports always state planned, executed and excluded counts separately.

**What a `FindingContract` failure means.** Cisco checks every finding of its bundled Python analyzers against a
declared rule list. When a finding uses a rule ID that is not declared, Cisco (in `core/scanner.py`) keeps the
finding visible, marks it with `metadata.rule_contract.status = "invalid"`, excludes it from its policy (CEL) gating,
sets `scan_metadata.rule_contract.status = "failed"`, and appends `FindingContract:<rule>:<code>` to
`analyzers_failed`. We do not edit Cisco and we do not drop the finding.

When the code is `UNKNOWN_BUNDLED_PYTHON_RULE` (a rule Cisco emits but never registered, which in 2.3.1 happens
for `BEHAVIOR_CROSSFILE_ENV_VAR_EXFILTRATION`) and nothing else went wrong, the scan is `CONTRACT_ONLY`. This
separates a known, possibly frequent Cisco bookkeeping issue from real analyzer failures. **`CONTRACT_ONLY` is not
treated as complete or accurate**: Cisco flagged its own output as inconsistent, and we cannot verify from outside
that nothing else was affected. That is why these scans stay out of the primary analysis and only appear in a
sensitivity analysis that shows whether including them would change a conclusion. The recognised codes are listed
in `KNOWN_CONTRACT_CODES`; any other contract code makes the scan `PARTIAL`.

Older saved reports have no run record, so their exit code is "not recorded" rather than assumed.

**Output:** `scan_runs_df` (one row per skill × configuration) and `reports` (parsed JSON of complete,
contract-only and partial scans).

In [ ]:
STATUS_OUTCOME = {"COMPLETE": "complete", "CONTRACT_ONLY": "contract_only", "PARTIAL": "partial",
                  "SKIPPED_INCOMPLETE_FOLDER": "excluded_incomplete_folder", "MISSING_REPORT": "missing"}
OUTCOME_ORDER = ["complete", "contract_only", "partial", "failed", "excluded_incomplete_folder", "missing"]
EXECUTED_OUTCOMES = ("complete", "contract_only", "partial", "failed")   # outcomes for which Cisco was actually invoked
NOT_EXECUTED_STATUSES = ("SKIPPED_INCOMPLETE_FOLDER", "MISSING_REPORT", "EXPORT_ERROR")
READABLE_STATUSES = ("COMPLETE", "CONTRACT_ONLY", "PARTIAL")   # scans whose findings are read into findings_df

# Cisco FindingContract codes we recognise as the known rule-registration issue (a rule that is emitted but not
# declared in Cisco's own registry). Any other code is treated as a real analyzer problem (PARTIAL).
KNOWN_CONTRACT_CODES = {"UNKNOWN_BUNDLED_PYTHON_RULE"}


def outcome_of(status):
    return STATUS_OUTCOME.get(status, "failed")


def scan_was_executed(status):
    """True when Cisco was invoked for this configuration; exclusions, missing reports and export errors were not scanned."""
    return status not in NOT_EXECUTED_STATUSES


def write_text_atomic(path, text):
    """Write text to a temporary file in the same folder, then rename: never a half-written file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile("w", encoding="utf-8", dir=path.parent, prefix=".tmp_", suffix=path.suffix, delete=False) as tmp:
        tmp.write(text)
    Path(tmp.name).replace(path)


def run_record_path(report_path):
    return report_path.with_name(report_path.stem + ".run.json")


def validate_report(obj):
    """Return None if `obj` looks like a Cisco single-skill report, otherwise a short reason."""
    if not isinstance(obj, dict):
        return "top level is not a JSON object"
    for key, expected_type in (("findings", list), ("is_safe", bool), ("max_severity", str)):
        if not isinstance(obj.get(key), expected_type):
            return f"missing or wrong type for '{key}'"
    return None


def is_known_contract_issue(entry):
    """True for 'FindingContract:<rule>:<codes>' entries whose codes are all in KNOWN_CONTRACT_CODES."""
    error = str(entry.get("error", ""))
    if not error.startswith("FindingContract:"):
        return False
    codes = {code.strip() for code in error.rsplit(":", 1)[-1].split(",") if code.strip()}
    return bool(codes) and codes <= KNOWN_CONTRACT_CODES


def failure_type(entry):
    """Short label for one analyzers_failed entry."""
    error = str(entry.get("error", ""))
    if is_known_contract_issue(entry):
        return "finding contract: unregistered rule (known Cisco issue)"
    if error.startswith("FindingContract:"):
        return "finding contract: other violation"
    if entry.get("analyzer") == "skill_loader":
        return "skill loader"
    return "analyzer error"


def assess_report(report, cfg, exit_code=None):
    """Classify one parsed report. Returns (status, reasons, analyzer_failures). Used for new AND saved reports."""
    problem = validate_report(report)
    if problem:
        return "FAILED_INVALID_FORMAT", [problem], []
    contract_reasons, other_reasons, failures = [], [], []      # known contract issue vs everything else
    for entry in report.get("analyzers_failed") or []:
        error = str(entry.get("error", ""))
        failures.append({"analyzer": entry.get("analyzer"), "error": entry.get("error"), "type": failure_type(entry),
                         "rule_id": error.split(":")[1] if error.startswith("FindingContract:") else None})
        reason = f"Cisco reported an analyzer failure ({entry.get('analyzer')}: {error})"
        (contract_reasons if is_known_contract_issue(entry) else other_reasons).append(reason)
    contract = (report.get("scan_metadata") or {}).get("rule_contract") or {}
    if contract.get("status") == "failed" and not any(str(f["error"]).startswith("FindingContract:") for f in failures):
        other_reasons.append(f"Cisco rule-contract check failed for {contract.get('invalid_findings')} finding(s)")
    not_run = sorted(set(cfg["required_analyzers"]) - set(report.get("analyzers_used") or []))
    if not_run:
        other_reasons.append(f"requested analyzer(s) missing from analyzers_used: {not_run}")
    if any(isinstance(f, dict) and f.get("rule_id") == "LLM_ANALYSIS_FAILED" for f in report["findings"]):
        other_reasons.append("the LLM analyzer could not review the skill (LLM_ANALYSIS_FAILED)")
    if exit_code not in (None, 0):
        other_reasons.append(f"scanner exit code {exit_code}")
    if other_reasons:
        status = "PARTIAL"
    elif contract_reasons:
        status = "CONTRACT_ONLY"
    else:
        status = "COMPLETE"
    return status, contract_reasons + other_reasons, failures

In [ ]:
def execute_scan(folder, cfg, report_path):
    """Run Cisco once and save its raw output plus a run record. Classification happens later, in load_and_assess."""
    command = [SCANNER, "scan", str(folder), *COMMON_ARGS, *cfg["extra_args"]]
    record = {"configuration": cfg["configuration"], "command": command, "scanner_version": INSTALLED_SCANNER_VERSION,
              "started_at": now_utc(), "report_file": report_path.name}
    started = time.perf_counter()
    try:
        proc = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, timeout=SCAN_TIMEOUT_SECONDS, check=False)
    except subprocess.TimeoutExpired:
        record.update(status="FAILED_TIMEOUT", detail=f"no result within {SCAN_TIMEOUT_SECONDS} s", exit_code=None,
                      wall_seconds=round(time.perf_counter() - started, 2))
    else:
        record.update(exit_code=proc.returncode, wall_seconds=round(time.perf_counter() - started, 2),
                      stderr_tail=proc.stderr.strip()[-1000:])
        try:
            json.loads(proc.stdout)
        except json.JSONDecodeError:
            write_text_atomic(report_path.with_name(report_path.stem + ".stdout.txt"), proc.stdout)
            record.update(status="FAILED_SCANNER_ERROR", detail=f"exit code {proc.returncode}; no JSON on standard output")
        else:
            write_text_atomic(report_path, proc.stdout)            # Cisco's JSON exactly as printed
            record["report_sha256"] = sha256_of_file(report_path)  # lets a later reanalysis detect edits
    write_text_atomic(run_record_path(report_path), json.dumps(record, indent=2) + "\n")


def record_skip(report_path, cfg, export):
    record = {"configuration": cfg["configuration"], "status": "SKIPPED_INCOMPLETE_FOLDER", "recorded_at": now_utc(),
              "detail": f"text missing for {export['files_missing']} file(s): {export['missing_files'][:5]}"}
    write_text_atomic(run_record_path(report_path), json.dumps(record, indent=2) + "\n")


# Scanner version of saved reports from the old layout (they have no run records): taken from the saved run_metadata.json.
_source_scanner = SOURCE_METADATA.get("scanner") or {}
_source_versions = _source_scanner.get("versions_that_produced_reports") or []
SOURCE_SCANNER_VERSION = _source_scanner.get("version") or (_source_versions[0] if len(_source_versions) == 1 else None)


def load_and_assess(skill_dir, cfg, export):
    """Classify one configuration of one skill from the files on disk (identical for new and saved runs)."""
    report_path = skill_dir / "reports" / cfg["report_file"]
    run_record = {}
    if run_record_path(report_path).is_file():
        run_record = json.loads(run_record_path(report_path).read_text(encoding="utf-8"))
    result = {"status": None, "detail": "", "failures": [], "report": None, "report_sha256": None,
              "exit_code": run_record.get("exit_code"), "wall_seconds": run_record.get("wall_seconds"),
              "scanner_version": run_record.get("scanner_version") or SOURCE_SCANNER_VERSION or "not recorded",
              "version_source": "run record" if run_record.get("scanner_version") else
                                ("saved run_metadata.json" if SOURCE_SCANNER_VERSION else "not recorded")}

    if report_path.is_file():
        result["report_sha256"] = sha256_of_file(report_path)
        if run_record.get("report_sha256") and run_record["report_sha256"] != result["report_sha256"]:
            result.update(status="FAILED_INTEGRITY", detail="report file no longer matches the hash recorded at scan time")
        else:
            try:
                report = json.loads(report_path.read_text(encoding="utf-8"))
            except json.JSONDecodeError as exc:
                result.update(status="FAILED_INVALID_JSON", detail=str(exc))
            else:
                status, reasons, failures = assess_report(report, cfg, run_record.get("exit_code"))
                exit_note = (f"exit code {run_record['exit_code']}" if "exit_code" in run_record
                             else "exit code not recorded (saved before run records existed)")
                result.update(status=status, failures=failures, detail="; ".join(reasons) or exit_note,
                              report=report if status in READABLE_STATUSES else None)
    elif run_record.get("status"):
        result.update(status=run_record["status"], detail=run_record.get("detail", ""))
    elif run_record:
        result.update(status="MISSING_REPORT", detail="a run record exists but its report file is missing")
    elif cfg["scope"] == "Full folder" and export["folder_complete"] is False and not ALLOW_INCOMPLETE_FOLDERS:
        result.update(status="SKIPPED_INCOMPLETE_FOLDER", detail="no full-folder report and the folder is incomplete")
    else:
        result.update(status="MISSING_REPORT", detail="no report and no run record for this configuration")
    if result["report"] is None:            # no report was produced, so no scanner version produced it
        result.update(scanner_version=run_record.get("scanner_version"), version_source=None)
    result["outcome"] = outcome_of(result["status"])
    return result

In [ ]:
SCAN_RUN_COLUMNS = ["skill_id", "repo", "skill_path", "sample_stratum", "configuration", "scope", "analyzer_mode", "status", "outcome",
                    "scan_executed", "status_detail", "analyzer_failure_count", "analyzer_failure_types", "analyzers_failed",
                    "rule_contract_status", "required_analyzers", "analyzers_used", "exit_code", "wall_seconds", "scan_seconds",
                    "is_safe", "max_severity", "findings_count", "folder_complete", "files_missing", "frontmatter_valid",
                    "policy_name", "policy_preset", "policy_fingerprint", "scanner_version", "scanner_version_source",
                    "report_timestamp", "report_sha256", "report_file"]

scan_runs, reports = [], {}     # reports: (skill_id, configuration) -> (report dict, folder that was scanned)

for number, export in enumerate(exports, start=1):
    skill_dir = RUN_DIR / export["skill_id"]
    print(f"\n[{number}/{len(exports)}] {export['repo']} / {export['skill_path']}")
    for cfg in SCAN_CONFIGS:
        report_path = skill_dir / "reports" / cfg["report_file"]
        if RUN_MODE == "scan":
            if cfg["scope"] == "Full folder" and not export["folder_complete"] and not ALLOW_INCOMPLETE_FOLDERS:
                record_skip(report_path, cfg, export)
            else:
                execute_scan(skill_dir / SCOPE_FOLDER[cfg["scope"]], cfg, report_path)
        result = load_and_assess(skill_dir, cfg, export)      # same classification for new and saved reports

        row = dict.fromkeys(SCAN_RUN_COLUMNS)
        row.update(skill_id=export["skill_id"], repo=export["repo"], skill_path=export["skill_path"],
                   sample_stratum=export["sample_stratum"], configuration=cfg["configuration"], scope=cfg["scope"], analyzer_mode=cfg["analyzer_mode"],
                   status=result["status"], outcome=result["outcome"], scan_executed=scan_was_executed(result["status"]),
                   status_detail=result["detail"],
                   analyzer_failure_count=len(result["failures"]),
                   analyzer_failure_types="; ".join(sorted({f["type"] for f in result["failures"]})) or None,
                   analyzers_failed=json.dumps(result["failures"]) if result["failures"] else None,
                   required_analyzers="|".join(cfg["required_analyzers"]),
                   exit_code=result["exit_code"], wall_seconds=result["wall_seconds"],
                   folder_complete=export["folder_complete"], files_missing=export["files_missing"],
                   frontmatter_valid=export["frontmatter_valid"], scanner_version=result["scanner_version"],
                   scanner_version_source=result["version_source"], report_sha256=result["report_sha256"],
                   report_file=show_path(report_path))
        report = result["report"]
        if report is not None:
            meta = report.get("scan_metadata") or {}
            row.update(is_safe=report["is_safe"], max_severity=report["max_severity"], findings_count=len(report["findings"]),
                       scan_seconds=report.get("scan_duration_seconds"), report_timestamp=report.get("timestamp"),
                       analyzers_used="|".join(report.get("analyzers_used") or []),
                       rule_contract_status=(meta.get("rule_contract") or {}).get("status"),
                       policy_name=meta.get("policy_name"), policy_preset=meta.get("policy_preset_base"),
                       policy_fingerprint=meta.get("policy_fingerprint_sha256"))
            reports[(export["skill_id"], cfg["configuration"])] = (report, skill_dir / SCOPE_FOLDER[cfg["scope"]])
        scan_runs.append(row)
        shown = f"{row['findings_count']} findings, max {row['max_severity']}, is_safe={row['is_safe']}" if report is not None else ""
        print(f"    {cfg['configuration']:<28} {row['status']:<26} {shown}")
        if row["outcome"] != "complete":
            print(f"        why: {row['status_detail'][:220]}")

for err in export_errors:
    row = dict.fromkeys(SCAN_RUN_COLUMNS)
    row.update(skill_id=err["skill_id"], repo=err["repo"], skill_path=err["skill_path"], configuration="(export)",
               sample_stratum=err.get("sample_stratum"),
               status="EXPORT_ERROR", outcome="failed", scan_executed=False, status_detail=err["error"])
    scan_runs.append(row)

scan_runs_df = pd.DataFrame(scan_runs, columns=SCAN_RUN_COLUMNS)
scan_runs_df["exit_code"] = scan_runs_df["exit_code"].astype("Int64")


def run_counts(runs=None, n_skills=None):
    """Planned configurations vs executed Cisco scans vs exclusions. planned = executed + excluded + missing + export errors."""
    runs = scan_runs_df if runs is None else runs
    configs = runs[runs["configuration"] != "(export)"]
    export_failed = int(runs["status"].eq("EXPORT_ERROR").sum())
    executed = configs[configs["status"].map(scan_was_executed)]
    counts = {"planned": len(configs) + export_failed * len(SCAN_CONFIGS),
              "executed": len(executed),
              "excluded_incomplete_folder": int(configs["status"].eq("SKIPPED_INCOMPLETE_FOLDER").sum()),
              "missing": int(configs["status"].eq("MISSING_REPORT").sum()),
              "not_run_export_error": export_failed * len(SCAN_CONFIGS),
              "executed_outcomes": executed["outcome"].value_counts().reindex(list(EXECUTED_OUTCOMES), fill_value=0).to_dict()}
    parts = counts["executed"] + counts["excluded_incomplete_folder"] + counts["missing"] + counts["not_run_export_error"]
    assert parts == counts["planned"], f"planned {counts['planned']} != executed + excluded + missing + export errors ({parts})"
    return counts
scan_runs_df["findings_count"] = scan_runs_df["findings_count"].astype("Int64")

if scan_runs_df.empty:
    print("No scans (no skills selected).")
else:
    COUNTS = run_counts()
    print(f"\nConfigurations planned: {COUNTS['planned']}   Cisco scans executed: {COUNTS['executed']}   "
          f"excluded (incomplete folder): {COUNTS['excluded_incomplete_folder']}   missing reports: {COUNTS['missing']}   "
          f"not run (export error): {COUNTS['not_run_export_error']}")
    print("Outcomes of the executed scans:", ", ".join(f"{k} {v}" for k, v in COUNTS["executed_outcomes"].items()))
    fingerprints = scan_runs_df["policy_fingerprint"].dropna().unique()
    if len(fingerprints) > 1:
        print("WARNING: reports carry different policy fingerprints; results are not directly comparable:", fingerprints)

## 9 — Turn the reports into one findings table, and check each finding's evidence

A Cisco report is a nested JSON object: scan-level fields plus a list of findings, each with its own `metadata`
dictionary. Analysis needs a flat table with **one row per finding**, built from every `COMPLETE`,
`CONTRACT_ONLY` and `PARTIAL` report (the `run_status` column says which). Cisco's own values are copied unchanged: `rule_id`, `category`
(as `cisco_category`), `severity`, `analyzer`, `title`, `description`, `remediation`, Cisco's `finding_id`, and the
useful parts of `metadata` (matched text and regex, files involved and `evidence` for cross-file rules, code-block
flags, taxonomy codes, the per-finding `rule_contract` status) plus the complete metadata as JSON.

### Paths and line numbers: what we keep and what we add

* **`reported_file_path` and `reported_line_number` are exactly what Cisco wrote.** They are never changed.
* **`file_path`** is a normalized copy used for comparison and display. Some behavioral findings carry an
  absolute path of the folder that was scanned, and that folder moves when a run is copied for reanalysis. The
  notebook makes such paths relative to the scanned folder Cisco recorded in the report (`skill_path`), so the
  same file has the same key on every machine and in every run. `path_note` says what was done.
* **Why line numbers can be off.** Cisco reports positions in its own coordinate systems. In 2.3.1, findings from
  code blocks inside Markdown (`MDBLOCK_*`, behavioral analyzer) are numbered from the opening fence line, with
  the first code line counted as +0, and for `SKILL.md` they are counted inside the instruction body *after the
  YAML frontmatter is removed* (`behavioral_analyzer.py`, `_analyze_markdown_code_blocks`). Those reported lines
  point above the code they describe. Masked secrets in snippets (`post****`) also prevent naive text matching.

So instead of trusting or "fixing" the line, the notebook adds a separate, conservative **evidence check**. It
takes Cisco's evidence text (the `snippet`, or the metadata `matched_text` when there is no snippet) and looks for
it in the exported file. Whitespace runs are collapsed and Cisco's `****` masks match any text.

| `evidence_status` | meaning |
|---|---|
| `exact` | the evidence text is on the reported line |
| `ambiguous` | the evidence is not on the reported line but occurs on several other lines, or it is too short (fewer than 6 literal characters) to identify a line |
| `mismatch` | the evidence is not on the reported line and occurs on exactly one other line or nowhere, or the reported line is past the end of the file |
| `unavailable` | nothing to check: no file, no line number, no evidence text, or the file is not in the exported folder |

`evidence_candidate_lines` lists the lines where the evidence occurs. It is a pointer for the human reviewer,
**never** a replacement for `reported_line_number`. For multi-line snippets only the first line is checked, and
`evidence_detail` says so.

### Our own classifications (research decisions, not Cisco's)

* **`finding_kind`** separates `security` findings from `metadata_quality` ones (missing license, vague
  description, name/length limits) and `scan_coverage` notes (binary files, low analyzability, load fallbacks,
  analysis-incomplete warnings). The lists are explicit below; everything not listed counts as security.
* **`research_category`** maps Cisco's category (with a few rule-level overrides) onto the eight behavior
  categories of RQ1-B. Cisco has no rule family for account modification / sensitive external actions; that
  category stays visible with zero findings instead of being silently merged.

### Comparison keys

`location_key = rule | file_path | reported_line_number` and
`finding_key = location_key | hash(evidence text)`. Section 12 explains how these are used.

**Snippets in the CSV files are redacted** where they look like literal secrets (quoted passwords/tokens, URL
credentials, provider key prefixes); `snippet_redacted` says when that happened. Cisco's raw reports keep the
original text.

**Output:** `findings_df`.

In [ ]:
# --- Our classification of Cisco rules (edit here; these are research decisions, not Cisco's) -------------
METADATA_QUALITY_RULES = {
    "MANIFEST_MISSING_LICENSE", "MANIFEST_INVALID_NAME", "MANIFEST_DESCRIPTION_TOO_LONG", "MANIFEST_METADATA_INCOMPLETE",
    "MISSING_STRUCTURED_METADATA", "INVALID_STRUCTURED_METADATA", "MALFORMED_YAML_FRONTMATTER",
    "SKILL_METADATA_SIZE_LIMIT_EXCEEDED", "SOCIAL_ENG_VAGUE_DESCRIPTION",
    "TRIGGER_VAGUE_DESCRIPTION", "TRIGGER_OVERLY_GENERIC", "TRIGGER_DESCRIPTION_TOO_SHORT",
}
SCAN_COVERAGE_RULES = {
    "LOW_ANALYZABILITY", "UNANALYZABLE_BINARY", "BINARY_FILE_DETECTED", "ARCHIVE_FILE_DETECTED", "PYCACHE_FILES_DETECTED",
    "OVERSIZED_FILE", "EXCESSIVE_FILE_COUNT", "SKILL_LOAD_FALLBACK_USED", "SKILL_LOAD_FAILED", "SKILL_LOAD_REJECTED_LIMIT",
    "SYMLINK_NOT_SCANNED", "UNICODE_ANALYSIS_INCOMPLETE", "BEHAVIOR_DATAFLOW_ANALYSIS_INCOMPLETE",
    "BYTECODE_ANALYSIS_UNAVAILABLE", "ARCHIVE_EXTRACTION_FAILED",
    "LLM_ANALYSIS_FAILED", "LLM_CONTEXT_BUDGET_EXCEEDED", "META_CONTEXT_BUDGET_EXCEEDED",
}

RESEARCH_CATEGORIES = [
    "Credential / secret access or exposure",
    "External network communication / data exfiltration",
    "Command execution / dangerous shell operations",
    "File-system access or modification",
    "Prompt injection / instruction manipulation",
    "Excessive permissions / privilege escalation",
    "Supply-chain / dependency risks",
    "Account modification / sensitive external actions",
]
# Cisco category -> research category. Categories not listed become "Other (<cisco category>)".
RESEARCH_CATEGORY_BY_CISCO_CATEGORY = {
    "hardcoded_secrets": RESEARCH_CATEGORIES[0],
    "data_exfiltration": RESEARCH_CATEGORIES[1],
    "command_injection": RESEARCH_CATEGORIES[2],
    "prompt_injection": RESEARCH_CATEGORIES[4],
    "unicode_steganography": RESEARCH_CATEGORIES[4],
    "social_engineering": RESEARCH_CATEGORIES[4],
    "unauthorized_tool_use": RESEARCH_CATEGORIES[5],
    "autonomy_abuse": RESEARCH_CATEGORIES[5],
    "tool_chaining_abuse": RESEARCH_CATEGORIES[5],
    "transitive_trust_abuse": RESEARCH_CATEGORIES[5],
    "skill_discovery_abuse": RESEARCH_CATEGORIES[5],
    "supply_chain_attack": RESEARCH_CATEGORIES[6],
}
# Rule-ID prefixes that override the category mapping (checked first, longest match wins).
RESEARCH_CATEGORY_BY_RULE_PREFIX = {
    "SECRET_": RESEARCH_CATEGORIES[0],                        # hardcoded secrets
    "DATA_EXFIL_SENSITIVE_FILE": RESEARCH_CATEGORIES[0],      # reads ~/.ssh, .env, credential files (Cisco files it under exfiltration)
    "DATA_EXFIL_ENV": RESEARCH_CATEGORIES[0],                 # environment-variable dumps
    "BEHAVIOR_CREDENTIAL_FILE_ACCESS": RESEARCH_CATEGORIES[0],
    "BEHAVIOR_ENV_VAR_HARVESTING": RESEARCH_CATEGORIES[0],
    "PATH_TRAVERSAL": RESEARCH_CATEGORIES[3],                 # path traversal is the closest Cisco gets to file-system abuse
    "GLOB_HIDDEN_FILE_TARGETING": RESEARCH_CATEGORIES[3],
    "PIPELINE_TAINT_FLOW": RESEARCH_CATEGORIES[2],
    "MDBLOCK_PYTHON_SUBPROCESS": RESEARCH_CATEGORIES[2],
    "MDBLOCK_PYTHON_EVAL_EXEC": RESEARCH_CATEGORIES[2],
    "BEHAVIOR_EVAL_SUBPROCESS": RESEARCH_CATEGORIES[2],
    "BEHAVIOR_BASH_TAINT_FLOW": RESEARCH_CATEGORIES[2],
    "SUPPLY_CHAIN_": RESEARCH_CATEGORIES[6],
    "TOOL_ABUSE_SYSTEM_PACKAGE_INSTALL": RESEARCH_CATEGORIES[6],
    "TOOL_ABUSE_UNTRUSTED_PACKAGE_SOURCE": RESEARCH_CATEGORIES[6],
}
RESEARCH_CATEGORIES_NOT_COVERED_BY_CISCO = [RESEARCH_CATEGORIES[7]]   # no Cisco rule family maps here; documented gap


def classify_finding_kind(rule_id):
    if rule_id in METADATA_QUALITY_RULES:
        return "metadata_quality"
    if rule_id in SCAN_COVERAGE_RULES:
        return "scan_coverage"
    return "security"


def research_category(rule_id, cisco_category):
    for prefix in sorted(RESEARCH_CATEGORY_BY_RULE_PREFIX, key=len, reverse=True):
        if str(rule_id).startswith(prefix):
            return RESEARCH_CATEGORY_BY_RULE_PREFIX[prefix]
    return RESEARCH_CATEGORY_BY_CISCO_CATEGORY.get(cisco_category, f"Other ({cisco_category})")


# --- Redaction for exported/displayed snippets (Cisco's raw reports keep the original text) ------------------
SECRET_PATTERNS = [
    # key = "literal" or "key": "literal"  -> literal replaced; code such as os.environ["KEY"] stays readable
    (re.compile(r"(?i)((?:password|passwd|secret|token|api[_-]?key|access[_-]?key)\w*['\"]?\s*[=:]\s*)(['\"])([^'\"\s]{4,})\2"), r"\1\2[REDACTED]\2"),
    # KEY=value without quotes (.env / shell style), but not $VARIABLES or function calls
    (re.compile(r"(?i)((?:password|passwd|secret|token|api[_-]?key)\w*\s*=\s*)(?![$%{(])([A-Za-z0-9_\-/+]{8,})(?![\w.(\[])"), r"\1[REDACTED]"),
    (re.compile(r"(://[^/\s:@]+:)[^@\s]+@"), r"\1[REDACTED]@"),                                # user:password@host
    (re.compile(r"\b(AKIA|ASIA|ghp_|gho_|github_pat_|xox[bap]-|AIza|sk-)[A-Za-z0-9_\-]{8,}"), r"\1[REDACTED]"),
]


def redact(text):
    if not isinstance(text, str):
        return text
    for pattern, replacement in SECRET_PATTERNS:
        text = pattern.sub(replacement, text)
    return text


def shorten(text, max_len=100):
    """One-line, length-limited version of a text for tables."""
    if not isinstance(text, str):
        return text
    text = text.replace("\n", " ")
    return text if len(text) <= max_len else text[:max_len] + "..."

In [ ]:
def normalize_file_path(raw_path, report_root, scanned_folder):
    """Return (path relative to the scanned folder or None, note). The raw Cisco value is stored separately."""
    if not raw_path:
        return None, "no file reported"
    path = Path(raw_path)
    if not path.is_absolute():
        return path.as_posix(), "relative path as reported"
    for base in (report_root, scanned_folder):
        if not base:
            continue
        for candidate_base, candidate_path in ((Path(base), path), (Path(base).resolve(), path.resolve())):
            try:
                return candidate_path.relative_to(candidate_base).as_posix(), "absolute path made relative to the scanned folder"
            except ValueError:
                pass
    return raw_path, "absolute path outside the scanned folder (kept as reported)"


MASK = re.compile(r"\*{4,}")       # Cisco masks secrets as "post****"


def collapse_whitespace(text):
    return re.sub(r"\s+", " ", text.strip())


def verify_evidence(file_path, reported_line, evidence, scanned_folder):
    """Look for Cisco's evidence text in the exported file. Returns (status, detail, candidate_lines).

    The reported line is only compared, never replaced.
    """
    if file_path is None or Path(file_path).is_absolute():
        return "unavailable", "no file (inside the scanned folder) reported", ""
    path = scanned_folder / file_path
    if not path.is_file():
        return "unavailable", "reported file is not in the exported folder", ""
    lines = [collapse_whitespace(line) for line in path.read_text(encoding="utf-8", errors="replace").splitlines()]
    has_line = reported_line is not None and not pd.isna(reported_line)
    if not isinstance(evidence, str) or not evidence.strip():
        where = f"; reported line {reported_line} exists" if has_line and 1 <= reported_line <= len(lines) else ""
        return "unavailable", "no evidence text" + where, ""

    evidence_lines = [line for line in evidence.splitlines() if line.strip()]
    first = re.sub(r"(\.\.\.|…)$", "", collapse_whitespace(evidence_lines[0])).strip()   # drop a truncation marker
    pieces = MASK.split(first)
    pattern = re.compile(".*?".join(re.escape(piece) for piece in pieces))
    literal_chars = sum(len(piece.strip()) for piece in pieces)
    candidates = [number for number, line in enumerate(lines, start=1) if pattern.search(line)]
    shown = ", ".join(map(str, candidates[:10])) + (" ..." if len(candidates) > 10 else "")
    note = " (first line of a multi-line snippet checked)" if len(evidence_lines) > 1 else ""

    if not has_line:
        return "unavailable", f"no line number reported; evidence occurs on {len(candidates)} line(s){note}", shown
    reported_line = int(reported_line)
    if not 1 <= reported_line <= len(lines):
        return "mismatch", f"reported line {reported_line} is past the end of the file ({len(lines)} lines){note}", shown
    if literal_chars < 6:
        return "ambiguous", f"evidence text too short to identify a line{note}", shown
    if reported_line in candidates:
        return "exact", f"evidence on the reported line{note}", shown
    if not candidates:
        return "mismatch", f"evidence text not found in the file{note}", shown
    if len(candidates) == 1:
        return "mismatch", f"evidence not on the reported line; it occurs only on line {candidates[0]}{note}", shown
    return "ambiguous", f"evidence not on the reported line; it occurs on {len(candidates)} other lines{note}", shown

In [ ]:
METADATA_FIELDS_TO_KEEP = ["matched_text", "matched_pattern", "signature_context", "scanner_category", "aitech", "aitech_name",
                           "aisubtech", "aisubtech_name", "files_involved", "evidence", "from_code_block", "block_language",
                           "deduped_rule_ids", "threat_type"]
FINDING_COLUMNS = (["skill_id", "repo", "skill_path", "sample_stratum", "configuration", "scope", "analyzer_mode", "run_status", "run_outcome",
                    "finding_id", "rule_id", "cisco_category", "severity", "analyzer", "title", "description",
                    "reported_file_path", "reported_line_number", "file_path", "path_note", "location",
                    "snippet", "snippet_redacted", "evidence_source", "evidence_status", "evidence_detail", "evidence_candidate_lines",
                    "remediation", "cisco_rule_contract", "finding_kind", "research_category", "location_key", "finding_key"]
                   + [f"meta_{name}" for name in METADATA_FIELDS_TO_KEEP] + ["metadata_json"])


def as_cell_value(value):
    """Lists/dicts become JSON text so they survive a CSV round trip; scalars pass through."""
    return json.dumps(value, sort_keys=True) if isinstance(value, (list, dict)) else value


def describe_location(file_path):
    if file_path is None:
        return "whole skill (no file)"
    return "SKILL.md" if file_path == "SKILL.md" else "supporting file"


def flatten_findings(report, run, scanned_folder):
    """One flat row per finding in `report`; `run` is the matching scan_runs row (identity + configuration)."""
    rows = []
    for finding in report["findings"]:
        if not isinstance(finding, dict):
            continue
        meta = finding.get("metadata") or {}
        raw_path, raw_line = finding.get("file_path"), finding.get("line_number")
        file_path, path_note = normalize_file_path(raw_path, report.get("skill_path"), scanned_folder)
        if finding.get("snippet"):
            evidence, evidence_source = finding["snippet"], "snippet"
        elif meta.get("matched_text"):
            evidence, evidence_source = meta["matched_text"], "matched_text"
        else:
            evidence, evidence_source = None, "none"
        status, detail, candidates = verify_evidence(file_path, raw_line, evidence, scanned_folder)
        location_key = f"{finding.get('rule_id')}|{file_path}|{raw_line}"
        snippet = finding.get("snippet")
        row = {
            "skill_id": run["skill_id"], "repo": run["repo"], "skill_path": run["skill_path"], "sample_stratum": run["sample_stratum"],
            "configuration": run["configuration"], "scope": run["scope"], "analyzer_mode": run["analyzer_mode"],
            "run_status": run["status"], "run_outcome": run["outcome"],
            "finding_id": finding.get("id"), "rule_id": finding.get("rule_id"), "cisco_category": finding.get("category"),
            "severity": finding.get("severity"), "analyzer": finding.get("analyzer"),
            "title": finding.get("title"), "description": redact(finding.get("description")),
            "reported_file_path": raw_path, "reported_line_number": raw_line,
            "file_path": file_path, "path_note": path_note, "location": describe_location(file_path),
            "snippet": redact(snippet), "snippet_redacted": isinstance(snippet, str) and redact(snippet) != snippet,
            "evidence_source": evidence_source, "evidence_status": status, "evidence_detail": detail,
            "evidence_candidate_lines": candidates,
            "remediation": finding.get("remediation"), "cisco_rule_contract": (meta.get("rule_contract") or {}).get("status"),
            "finding_kind": classify_finding_kind(finding.get("rule_id")),
            "research_category": research_category(finding.get("rule_id"), finding.get("category")),
            "location_key": location_key,
            "finding_key": f"{location_key}|{hashlib.sha256(str(evidence).encode()).hexdigest()[:10]}",
            "metadata_json": json.dumps(meta, sort_keys=True, default=str),
        }
        for name in METADATA_FIELDS_TO_KEEP:
            row[f"meta_{name}"] = as_cell_value(meta.get(name))
        rows.append(row)
    return rows


finding_rows = []
for run in scan_runs:
    if (run["skill_id"], run["configuration"]) in reports:
        report, scanned_folder = reports[(run["skill_id"], run["configuration"])]
        finding_rows.extend(flatten_findings(report, run, scanned_folder))
findings_df = pd.DataFrame(finding_rows, columns=FINDING_COLUMNS)

SEVERITY_ORDER = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "INFO"]
findings_df["severity"] = pd.Categorical(findings_df["severity"], categories=SEVERITY_ORDER, ordered=True)
findings_df["configuration"] = pd.Categorical(findings_df["configuration"], categories=CONFIG_ORDER, ordered=True)
findings_df["reported_line_number"] = findings_df["reported_line_number"].astype("Int64")

raw_total = sum(len(report["findings"]) for report, _ in reports.values())
print(f"{len(findings_df)} finding rows from {len(reports)} report(s) "
      f"(Cisco listed {raw_total} findings in those reports; the two numbers must be equal).")
assert len(findings_df) == raw_total, "Some findings were not flattened"
if not findings_df.empty:
    print("By run outcome:", findings_df["run_outcome"].value_counts().to_dict())
    print("By kind       :", findings_df["finding_kind"].value_counts().to_dict())
    print("Evidence check:", findings_df["evidence_status"].value_counts().to_dict())
    preview = findings_df[["skill_id", "configuration", "run_status", "severity", "rule_id", "file_path",
                           "reported_line_number", "evidence_status", "snippet"]].copy()
    preview["snippet"] = preview["snippet"].map(shorten)
    display(preview.sort_values(["skill_id", "configuration", "severity"]).reset_index(drop=True))

## 10 — Save the structured results

`scan_summary.csv` (one row per skill × configuration) and `individual_findings.csv` (one row per finding) are
written now; the comparison tables, the manual-review file, the figures and `results_summary.md` follow in later
sections, and `run_metadata.json` is written here and completed in Section 15. All of these files live in this
run's folder only. The raw Cisco reports stay under each `<skill_id>/reports/`; the CSV files never replace them.

In [ ]:
def run_file(name):
    """Path of an output file inside this run's folder (None when no run folder exists, e.g. search mode)."""
    return RUN_DIR / name if RUN_DIR is not None else None


def save_csv(df, name):
    path = run_file(name)
    if path is None:
        print(f"(not saved: no run folder) {name}")
        return None
    df.to_csv(path, index=False)
    print(f"saved {show_path(path)}  ({len(df)} rows)")
    return path


def build_run_metadata(stage):
    return {
        "run_id": RUN_ID_USED, "run_mode": RUN_MODE, "stage": stage,
        "run_started_at": RUN_STARTED_AT, "written_at": now_utc(),
        "source_run": None if SOURCE_DIR is None else {
            "path": show_path(SOURCE_DIR), "run_metadata_found": bool(SOURCE_METADATA),
            "scanner_version": SOURCE_SCANNER_VERSION,
            "skills": [d.name for d in SOURCE_SKILL_DIRS]},
        "database": {"path": show_path(DB_PATH), "tag": DB_TAG, "sha256": DB_SHA256, "available": conn is not None},
        "scanner": {"installed_version": INSTALLED_SCANNER_VERSION,
                    "versions_that_produced_reports": sorted(scan_runs_df["scanner_version"].dropna().unique().tolist()),
                    "policy_setting": POLICY, "common_args": COMMON_ARGS,
                    "policy_fingerprints_seen": sorted(scan_runs_df["policy_fingerprint"].dropna().unique().tolist()),
                    "configurations": SCAN_CONFIGS},
        "settings": {"SELECTION_MODE": SELECTION_MODE, "PILOT": PILOT, "EXACT_REPO": EXACT_REPO, "EXACT_PATH": EXACT_PATH,
                     "SEARCH_KEYWORD": SEARCH_KEYWORD, "MAX_SKILLS": MAX_SKILLS, "LENIENT": LENIENT,
                     "ALLOW_INCOMPLETE_FOLDERS": ALLOW_INCOMPLETE_FOLDERS, "SCAN_TIMEOUT_SECONDS": SCAN_TIMEOUT_SECONDS,
                     "ENABLE_OSV": ENABLE_OSV, "ENABLE_LLM_CONFIGS": ENABLE_LLM_CONFIGS},
        "environment": {"python": platform.python_version(), "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
                        "platform": platform.platform()},
        "sample": SAMPLE_INFO,
        "selection": [{"repo": r, "skill_path": p} for r, p in selections],
        "skills": exports_df.to_dict("records"),
        "export_errors": export_errors,
        "run_outcome_counts": scan_runs_df["outcome"].value_counts().to_dict(),
        "configuration_counts": {k: v for k, v in run_counts().items()},
        "run_status_counts": scan_runs_df["status"].value_counts().to_dict(),
        "finding_rows": int(len(findings_df)),
    }


def save_run_metadata(stage):
    path = run_file("run_metadata.json")
    if path is not None:
        path.write_text(json.dumps(build_run_metadata(stage), indent=2, default=str) + "\n", encoding="utf-8")
        print(f"saved {show_path(path)}  (stage: {stage})")


save_csv(scan_runs_df, "scan_summary.csv")
save_csv(findings_df, "individual_findings.csv")
save_run_metadata("tables saved; analysis not finished")

## 11 — Analysis A: what security findings does Cisco report?

**Primary tables use `COMPLETE` scans only.** Findings from `CONTRACT_ONLY` and `PARTIAL` scans are listed
separately at the end of this section (A8): they are real Cisco output, but Cisco reported a problem with the scan,
so they are not mixed into counts we would report as results. Contract-only scans also enter the sensitivity
analysis of Sections 12–13.

Two counts are always kept apart: the **number of findings** (rows) and the **number of distinct skills** with at
least one such finding, because one script can produce ten findings for the same rule.

* **A1** — per-skill verdict matrix (all scans with a report, with their outcome).
* **A2** — findings by kind: security vs metadata quality vs coverage notes.
* **A3** — security findings by Cisco category and by research category (findings and distinct skills).
* **A4** — the rules that fire most often.
* **A5** — severity per configuration.
* **A6** — where findings point: `SKILL.md`, a supporting file, or the whole skill.
* **A7** — metadata-quality and coverage findings, listed so nobody mistakes them for security results.
* **A8** — findings outside the primary group (contract-only and partial scans).

Chart colors follow a fixed order per configuration, so a configuration has the same color in every figure. The
category and severity charts are also saved as PNG files in the run's `figures/` folder.

In [ ]:
# Shared subsets and helpers for Sections 11-15 ---------------------------------------------------------------
complete_runs = scan_runs_df[scan_runs_df["outcome"] == "complete"]
PRIMARY_CONFIGS = [c for c in CONFIG_ORDER if c in set(complete_runs["configuration"])]
primary = findings_df[findings_df["run_outcome"] == "complete"]
security = primary[primary["finding_kind"] == "security"].copy()
non_primary = findings_df[findings_df["run_outcome"].isin(["contract_only", "partial"])
                          & (findings_df["finding_kind"] == "security")].copy()

SERIES_COLORS = dict(zip(CONFIG_ORDER, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
SEVERITY_COLORS = {"INFO": "#86b6ef", "LOW": "#5598e7", "MEDIUM": "#2a78d6", "HIGH": "#1c5cab", "CRITICAL": "#0d366b"}
SAVED_FIGURES = []


def count_table(df, rows, columns=None, unit="findings"):
    """Pivot `df` into counts. unit="findings" counts rows; unit="skills" counts distinct skill_id."""
    if df.empty:
        return pd.DataFrame()
    values, how = ("skill_id", "nunique") if unit == "skills" else ("finding_key", "count")
    table = df.pivot_table(index=rows, columns=columns, values=values, aggfunc=how, fill_value=0, observed=True)
    if table.index.nlevels == 1:
        table.index = table.index.astype(str)        # categorical -> plain labels
    table.columns = table.columns.astype(str)
    if columns == "configuration":
        # Keep every configuration that has a complete scan, even with zero findings, so "0" stays visible.
        table = table.reindex(columns=PRIMARY_CONFIGS, fill_value=0)
    return table.astype(int)


def grouped_bar_chart(table, title, xlabel, colors=None, save_as=None):
    """Horizontal grouped bars: one group per table row, one bar per column; counts written at the bar ends."""
    if table.empty or int(table.to_numpy().sum()) == 0:
        print(f"(no data to plot for: {title})")
        return None
    rows, series = list(table.index), list(table.columns)
    bar_h = 0.8 / len(series)
    peak = table.to_numpy().max()
    fig, ax = plt.subplots(figsize=(9, 0.9 + 0.45 * len(rows) * len(series)))
    for i, name in enumerate(series):
        y = [r - 0.4 + bar_h * (i + 0.5) for r in range(len(rows))]
        values = table[name].to_numpy()
        ax.barh(y, values, height=bar_h * 0.9, color=(colors or SERIES_COLORS).get(name, "#777777"), label=str(name))
        for yy, v in zip(y, values):
            if v:
                ax.text(v + peak * 0.01, yy, str(int(v)), va="center", fontsize=8, color="#444444")
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels([str(r)[:60] for r in rows], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, peak * 1.12)                                  # room for the count labels
    ax.set_title(title, loc="left", fontsize=11)
    ax.legend(fontsize=8, frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0))   # outside the plot
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(axis="x", color="#dddddd", linewidth=0.6)
    ax.set_axisbelow(True)
    plt.tight_layout()
    path = None
    if save_as and RUN_DIR is not None:
        path = RUN_DIR / "figures" / save_as
        fig.savefig(path, dpi=150, bbox_inches="tight")
        SAVED_FIGURES.append((path, title))
    plt.show()
    return path


print(f"Complete scans: {len(complete_runs)} of {len(scan_runs_df)} configuration run(s).")
print(f"Primary security findings (complete scans): {len(security)}  |  "
      f"security findings outside the primary group (contract-only + partial scans): {len(non_primary)}  |  "
      f"metadata/coverage findings (complete scans): {len(primary) - len(security)}")

In [ ]:
runs_with_report = scan_runs_df[scan_runs_df["outcome"].isin(["complete", "contract_only", "partial"])]
if runs_with_report.empty:
    print("No scans with a readable report: nothing to analyse. Check the statuses in Section 8.")
else:
    a1 = runs_with_report.pivot_table(index=["repo", "skill_path"], columns="configuration",
                                      values=["outcome", "findings_count", "max_severity", "is_safe"], aggfunc="first")
    a1 = a1.reindex(columns=[c for c in CONFIG_ORDER if c in set(runs_with_report["configuration"])], level=1)
    print("A1  Outcome, findings, maximum severity and is_safe per configuration (all finding kinds; outcome shown per scan)")
    display(a1)

    print("\nA2  Findings by kind (complete scans)")
    display(count_table(primary, "finding_kind", "configuration"))

In [ ]:
if security.empty:
    print("No security findings in complete scans.")
else:
    print("A3a  Security findings by Cisco category (number of findings, complete scans)")
    display(count_table(security, "cisco_category", "configuration"))
    print("\nA3b  Security findings by Cisco category (number of distinct skills with >= 1 finding)")
    display(count_table(security, "cisco_category", "configuration", unit="skills"))
    print("\nA3c  Security findings by research category (our mapping, see Section 9)")
    a3c = count_table(security, "research_category", "configuration")
    a3c = a3c.reindex(RESEARCH_CATEGORIES + [c for c in a3c.index if c not in RESEARCH_CATEGORIES], fill_value=0)
    display(a3c)
    print("Not covered by any Cisco rule family:", RESEARCH_CATEGORIES_NOT_COVERED_BY_CISCO)
    grouped_bar_chart(count_table(security, "cisco_category", "configuration"),
                      "Security findings by Cisco category and configuration (complete scans)", "number of findings",
                      save_as="findings_by_category.png")

In [ ]:
if not security.empty:
    a4 = (security.groupby(["rule_id", "cisco_category", "analyzer"], observed=True)
          .agg(findings=("finding_key", "size"), skills=("skill_id", "nunique"),
               configurations=("configuration", lambda s: ", ".join(sorted(set(map(str, s))))),
               severities=("severity", lambda s: ", ".join(sorted(set(map(str, s))))))
          .sort_values("findings", ascending=False).reset_index())
    print("A4  Cisco rules by number of security findings (and distinct skills), complete scans")
    display(a4)

    print("\nA5  Severity of security findings per configuration (complete scans)")
    a5 = count_table(security, "configuration", "severity")
    a5 = a5.reindex(index=[c for c in PRIMARY_CONFIGS], columns=[s for s in SEVERITY_ORDER if s in a5.columns], fill_value=0)
    display(a5)
    grouped_bar_chart(a5, "Severity of security findings per configuration (complete scans)", "number of findings",
                      colors=SEVERITY_COLORS, save_as="severity_by_configuration.png")

    print("\nA6  Where security findings point (SKILL.md, a supporting file, or the whole skill)")
    display(count_table(security, "location", "configuration"))
    supporting = security[security["location"] == "supporting file"]
    if not supporting.empty:
        print("\nA6b  Supporting files with security findings")
        display(supporting.groupby(["skill_id", "file_path"], observed=True)
                .agg(findings=("finding_key", "size"), rules=("rule_id", lambda s: ", ".join(sorted(set(s)))))
                .sort_values("findings", ascending=False))

In [ ]:
non_security = primary[primary["finding_kind"] != "security"]
if non_security.empty:
    print("A7  No metadata-quality or scan-coverage findings in complete scans.")
else:
    print("A7  Metadata-quality and scan-coverage findings (not security results)")
    display(non_security[["skill_id", "configuration", "finding_kind", "severity", "rule_id", "title", "file_path"]]
            .sort_values(["skill_id", "configuration"]).reset_index(drop=True))

if non_primary.empty:
    print("\nA8  No security findings outside the primary group (no contract-only or partial scans with findings).")
else:
    print("\nA8  Security findings from CONTRACT_ONLY and PARTIAL scans (kept, but excluded from A2-A6 and from primary comparisons)")
    a8 = non_primary[["skill_id", "configuration", "run_status", "severity", "rule_id", "cisco_category", "analyzer",
                      "file_path", "reported_line_number", "evidence_status", "cisco_rule_contract", "snippet"]].copy()
    a8["snippet"] = a8["snippet"].map(shorten)
    display(a8.sort_values(["skill_id", "configuration", "severity"]).reset_index(drop=True))
    print("Reasons:")
    for _, run in scan_runs_df[scan_runs_df["outcome"].isin(["contract_only", "partial"])].iterrows():
        print(f"  - {run['skill_id']} | {run['configuration']}: {run['status_detail']}")

## 12 — Analysis B: does the full skill folder change what Cisco finds? (RQ1-A)

For each skill and each analyzer mode we compare the `SKILL.md only` scan with the `Full folder` scan **finding
by finding**. Matching is one-to-one and runs in three passes, from strict to loose:

| change | rule |
|---|---|
| `unchanged` | same rule, file, reported line and evidence text, and the same severity |
| `changed` | same rule at the same file and line, but different evidence or severity |
| `possibly_equivalent` | same rule and file at a *different* line; may be the same behavior reported at another position, so check it manually |
| `additional` | only in the full-folder scan |
| `missing` | only in the SKILL.md-only scan |

Findings with **different rule IDs are never matched automatically**, even when they describe the same behavior
(for example a cross-file rule and a single-file rule about the same script). Those pairs show up as `additional`
and `missing` and are left for the manual review.

**Pair eligibility, primary and sensitivity analysis.** Every pair of scans gets a `pair_status`:

| `pair_status` | rule | used in |
|---|---|---|
| `eligible` | both scans `COMPLETE` (and, for context comparisons, a complete folder) | primary **and** sensitivity analysis |
| `sensitivity: contract-only scan` | both scans `COMPLETE` or `CONTRACT_ONLY`, at least one `CONTRACT_ONLY`, folder complete | sensitivity analysis only |
| `preliminary: partial scan` | at least one scan `PARTIAL` | neither; shown as preliminary |
| `excluded: incomplete folder` | the full-folder scan was deliberately not run because GitSkills did not store every supporting file (a design exclusion, not a scanner failure) | neither; counted as `excluded_incomplete_folder` |
| `preliminary: incomplete folder` | the full folder is incomplete but was scanned anyway (`ALLOW_INCOMPLETE_FOLDERS = True`) | neither; shown as preliminary, counted as `excluded_incomplete_folder` |
| `not comparable` | a scan failed or its report is missing | neither |

The **primary analysis** answers the research question with complete scans only. The **sensitivity analysis**
repeats it with the contract-only pairs added, so we can see whether Cisco's rule-registration issue would change a
conclusion; it does not assume those scans are complete or accurate. Every comparison reports its **sample size**
(pairs used) and the pairs excluded from the primary analysis, by reason. The comparisons use security findings only,
so a missing-license note never counts as "context changed the result".

* New findings located in supporting files show behavior `SKILL.md` alone could not reveal.
* New findings located *in `SKILL.md`* when the full folder is present (for example a rule such as
  `TOOL_ABUSE_UNDECLARED_NETWORK`) show Cisco re-reading the instructions in the light of the scripts: context
  changes the interpretation rather than adding an operation.
* Missing findings would mean extra context made Cisco drop a warning. Cisco's deterministic analyzers have no
  "explain away" logic, so inspect any such case manually before claiming it.

In [ ]:
def same_value(x, y):
    """Equality that treats two missing values as equal (pandas uses NaN/NA, and NaN != NaN)."""
    x_missing, y_missing = pd.isna(x), pd.isna(y)
    return (x_missing and y_missing) or (not x_missing and not y_missing and x == y)


MATCH_PASSES = [
    ("unchanged", lambda a, b: a["finding_key"] == b["finding_key"] and same_value(a["severity"], b["severity"])),
    ("changed", lambda a, b: a["location_key"] == b["location_key"]),
    ("possibly_equivalent", lambda a, b: a["rule_id"] == b["rule_id"] and same_value(a["file_path"], b["file_path"])),
]


def compare_findings(before, after):
    """One-to-one finding diff between two scans of the same skill (`before` -> `after`)."""
    before_rows = before.sort_values("reported_line_number", na_position="last").to_dict("records")
    after_rows = after.sort_values("reported_line_number", na_position="last").to_dict("records")
    after_change = [None] * len(after_rows)
    before_partner = [None] * len(before_rows)
    for change, matches in MATCH_PASSES:
        for ai, a in enumerate(after_rows):
            if after_change[ai] is not None:
                continue
            for bi, b in enumerate(before_rows):
                if before_partner[bi] is None and matches(a, b):
                    after_change[ai], before_partner[bi] = change, ai
                    a["matched_line_before"], a["matched_severity_before"] = b["reported_line_number"], b["severity"]
                    break
    rows = [{**a, "change": after_change[ai] or "additional"} for ai, a in enumerate(after_rows)]
    rows += [{**b, "change": "missing"} for bi, b in enumerate(before_rows) if before_partner[bi] is None]
    return rows


def pair_status(before_run, after_run, needs_complete_folder):
    outcomes = {before_run["outcome"], after_run["outcome"]}
    if "excluded_incomplete_folder" in outcomes:
        return "excluded: incomplete folder"
    if not outcomes <= {"complete", "contract_only", "partial"}:
        return f"not comparable ({before_run['status']} vs {after_run['status']})"
    if needs_complete_folder and after_run["folder_complete"] is not True:
        return "preliminary: incomplete folder"
    if "partial" in outcomes:
        return "preliminary: partial scan"
    if "contract_only" in outcomes:
        return "sensitivity: contract-only scan"
    return "eligible"


PRIMARY_PAIR_STATUSES = ["eligible"]
SENSITIVITY_PAIR_STATUSES = ["eligible", "sensitivity: contract-only scan"]


RUN_LOOKUP = {(r["skill_id"], r["configuration"]): r for r in scan_runs}
CHANGE_ORDER = ["unchanged", "changed", "possibly_equivalent", "additional", "missing"]
DIFF_COLUMNS = FINDING_COLUMNS + ["matched_line_before", "matched_severity_before", "change", "comparison", "pair_status"]


def paired_comparison(pairs, needs_complete_folder):
    """`pairs` lists (label, before configuration, after configuration). Returns (diff table, pair table)."""
    diff_rows, pair_rows = [], []
    for export in exports:
        for label, before_cfg, after_cfg in pairs:
            before_run, after_run = RUN_LOOKUP.get((export["skill_id"], before_cfg)), RUN_LOOKUP.get((export["skill_id"], after_cfg))
            if before_run is None or after_run is None:
                continue
            status = pair_status(before_run, after_run, needs_complete_folder)
            pair_rows.append({"comparison": label, "skill_id": export["skill_id"],
                              "before": f"{before_cfg} ({before_run['status']})", "after": f"{after_cfg} ({after_run['status']})",
                              "pair_status": status})
            if status.startswith(("not comparable", "excluded")):
                continue
            subset = findings_df[(findings_df["skill_id"] == export["skill_id"]) & (findings_df["finding_kind"] == "security")]
            for row in compare_findings(subset[subset["configuration"] == before_cfg], subset[subset["configuration"] == after_cfg]):
                diff_rows.append({**row, "comparison": label, "pair_status": status})
    return pd.DataFrame(diff_rows, columns=DIFF_COLUMNS), pd.DataFrame(pair_rows)


def pair_group(status):
    """Coarse reason label for sample-size tables."""
    if status == "eligible":
        return "complete"
    if status.startswith("sensitivity"):
        return "contract_only"
    if status == "preliminary: partial scan":
        return "partial"
    if status in ("preliminary: incomplete folder", "excluded: incomplete folder"):
        return "incomplete_folder"
    return "not_comparable"


def sample_sizes(pairs):
    """Per comparison: pairs available, n used by the primary and sensitivity analyses, pairs excluded by reason."""
    if pairs.empty:
        return pd.DataFrame()
    grouped = pairs.assign(group=pairs["pair_status"].map(pair_group))
    t = grouped.pivot_table(index="comparison", columns="group", values="skill_id", aggfunc="count", fill_value=0)
    t = t.reindex(index=list(dict.fromkeys(pairs["comparison"])),
                  columns=["complete", "contract_only", "partial", "incomplete_folder", "not_comparable"], fill_value=0)
    return pd.DataFrame({"pairs_total": t.sum(axis=1), "primary_n": t["complete"],
                         "sensitivity_n": t["complete"] + t["contract_only"],
                         "excluded_contract_only": t["contract_only"], "excluded_partial": t["partial"],
                         "excluded_incomplete_folder": t["incomplete_folder"],
                         "excluded_not_comparable": t["not_comparable"]}).astype(int)


def comparison_totals(diff, pairs, statuses):
    """Per comparison: pairs used, finding changes summed over those pairs, skills that gained findings."""
    if pairs.empty:
        return pd.DataFrame()
    rows = []
    for label in dict.fromkeys(pairs["comparison"]):
        used = pairs[(pairs["comparison"] == label) & pairs["pair_status"].isin(statuses)]
        d = diff[(diff["comparison"] == label) & diff["pair_status"].isin(statuses)]
        row = {"comparison": label, "pairs_used": len(used), "pairs_with_findings": d["skill_id"].nunique()}
        row.update({change: int((d["change"] == change).sum()) for change in CHANGE_ORDER})
        row["skills_with_additional"] = d.loc[d["change"] == "additional", "skill_id"].nunique()
        rows.append(row)
    return pd.DataFrame(rows).set_index("comparison")


def sensitivity_effect(diff, pairs):
    """Plain sentences: does adding contract-only pairs change each comparison?"""
    primary_t = comparison_totals(diff, pairs, PRIMARY_PAIR_STATUSES)
    sens_t = comparison_totals(diff, pairs, SENSITIVITY_PAIR_STATUSES)
    lines = []
    for label in primary_t.index:
        p, s = primary_t.loc[label], sens_t.loc[label]
        if s["pairs_used"] == p["pairs_used"]:
            lines.append(f"{label}: no contract-only pairs, so the sensitivity analysis equals the primary analysis (n = {p['pairs_used']}).")
            continue
        start = (f"{label}: the primary analysis has no usable pair (n = 0); only the sensitivity analysis (n = {s['pairs_used']}) "
                 "covers this comparison, so any conclusion here depends on contract-only scans"
                 if p["pairs_used"] == 0 else
                 f"{label}: primary n = {p['pairs_used']}, sensitivity n = {s['pairs_used']}")
        lines.append(f"{start}. Additional findings {p['additional']} vs {s['additional']}, missing {p['missing']} vs {s['missing']}, "
                     f"unchanged {p['unchanged']} vs {s['unchanged']}; skills gaining findings {p['skills_with_additional']} vs "
                     f"{s['skills_with_additional']} (primary vs sensitivity).")
    return lines


def change_counts(diff):
    if diff.empty:
        return pd.DataFrame()
    table = diff.pivot_table(index=["skill_id", "comparison"], columns="change", values="finding_key",
                             aggfunc="count", fill_value=0, observed=True)
    return table.reindex(columns=CHANGE_ORDER, fill_value=0).astype(int)

In [ ]:
CONTEXT_PAIRS = [(f"SKILL.md only -> Full folder: {mode}", f"SKILL.md only | {mode}", f"Full folder | {mode}")
                 for mode in dict.fromkeys(cfg["analyzer_mode"] for cfg in SCAN_CONFIGS)]
context_diff, context_pairs = paired_comparison(CONTEXT_PAIRS, needs_complete_folder=True)
context_eligible = context_diff[context_diff["pair_status"] == "eligible"]

print("B0  Sample sizes: pairs used by the primary and sensitivity analyses, and pairs excluded (by reason)")
display(sample_sizes(context_pairs) if not context_pairs.empty else pd.DataFrame({"note": ["no pairs"]}))

print("\nB1  PRIMARY analysis (complete scans only): finding changes summed over the pairs used")
display(comparison_totals(context_diff, context_pairs, PRIMARY_PAIR_STATUSES))
print("\nB2  SENSITIVITY analysis (complete + contract-only scans)")
display(comparison_totals(context_diff, context_pairs, SENSITIVITY_PAIR_STATUSES))
for line in sensitivity_effect(context_diff, context_pairs):
    print("   ", line)
if not context_eligible.empty:
    print("\nB2b Primary change counts per skill")
    display(change_counts(context_eligible))

In [ ]:
if not context_eligible.empty:
    shown = context_eligible[context_eligible["change"] != "unchanged"].copy()
    shown["snippet"] = shown["snippet"].map(shorten)
    print("B3  Every finding that is additional, changed, possibly equivalent or missing (eligible pairs)")
    display(shown[["skill_id", "comparison", "change", "severity", "rule_id", "cisco_category", "analyzer", "location",
                   "file_path", "reported_line_number", "matched_line_before", "evidence_status", "snippet"]]
            .sort_values(["skill_id", "comparison", "change", "file_path"]).reset_index(drop=True))

    additional = context_eligible[context_eligible["change"] == "additional"]
    if not additional.empty:
        print("\nB4  Files that contribute the additional findings (location = SKILL.md means context changed how the instructions were read)")
        display(additional.groupby(["comparison", "location", "file_path"], dropna=False, observed=True)
                .agg(findings=("finding_key", "size"), skills=("skill_id", "nunique"),
                     rules=("rule_id", lambda s: ", ".join(sorted(set(s))))).reset_index())

preliminary_context = context_diff[context_diff["pair_status"].str.startswith("preliminary")]
if not preliminary_context.empty:
    print("\nB5  PRELIMINARY context comparisons (a scan is partial or the folder incomplete; in neither analysis)")
    display(change_counts(preliminary_context))

if 0 < exports_df["skill_id"].nunique() <= 15 and not security.empty:
    grouped_bar_chart(count_table(security, "skill_id", "configuration"),
                      "Security findings per skill and configuration (complete scans)", "number of security findings")

## 13 — Analysis C: what does the behavioral analyzer add? (analyzer part of RQ1-C)

Same finding-level comparison, now holding the scope fixed and comparing `default` with `--use-behavioral`. The
behavioral analyzer only **adds** an analyzer (the default ones still run), so `missing` should be rare; when it
appears, Cisco's output de-duplication (which prefers the behavioral version of a finding at the same location)
is the likely reason, and the pair usually shows up as `changed`. Folder completeness does not matter here
because both scans see the same folder, so only scan outcomes decide eligibility.

* **C0** — pair eligibility.
* **C1** — change counts per skill and scope (eligible pairs).
* **C2** — the findings only behavioral analysis produced, with their `analyzer` label.
* **C3** — security findings by Cisco category and analyzer label (complete scans).

The section ends with one chart that summarises **both** comparisons (Sections 12 and 13) for eligible pairs; it
is saved as `figures/configuration_comparisons.png`.

In [ ]:
ANALYZER_PAIRS = [(f"default -> {mode}: {scope}", f"{scope} | default", f"{scope} | {mode}")
                  for mode in ("behavioral", "llm") for scope in ("SKILL.md only", "Full folder")
                  if f"{scope} | {mode}" in CONFIG_BY_NAME]
analyzer_diff, analyzer_pairs = paired_comparison(ANALYZER_PAIRS, needs_complete_folder=False)
analyzer_eligible = analyzer_diff[analyzer_diff["pair_status"] == "eligible"]

print("C0  Sample sizes: pairs used by the primary and sensitivity analyses, and pairs excluded (by reason)")
display(sample_sizes(analyzer_pairs) if not analyzer_pairs.empty else pd.DataFrame({"note": ["no pairs"]}))

print("\nC1  PRIMARY analysis (complete scans only): finding changes summed over the pairs used")
display(comparison_totals(analyzer_diff, analyzer_pairs, PRIMARY_PAIR_STATUSES))
print("\nC1s SENSITIVITY analysis (complete + contract-only scans)")
display(comparison_totals(analyzer_diff, analyzer_pairs, SENSITIVITY_PAIR_STATUSES))
for line in sensitivity_effect(analyzer_diff, analyzer_pairs):
    print("   ", line)
if not analyzer_eligible.empty:
    print("\nC1b Primary change counts per skill")
    display(change_counts(analyzer_eligible))
    added = analyzer_eligible[analyzer_eligible["change"] != "unchanged"].copy()
    added["snippet"] = added["snippet"].map(shorten)
    print("\nC2  Findings that differ between default and behavioral analysis (eligible pairs)")
    display(added[["skill_id", "comparison", "change", "severity", "rule_id", "cisco_category", "analyzer", "file_path",
                   "reported_line_number", "evidence_status", "snippet"]].sort_values(["skill_id", "comparison", "file_path"])
            .reset_index(drop=True))

if not security.empty:
    print("\nC3  Security findings by Cisco category and the analyzer that emitted them (complete scans)")
    display(count_table(security, "cisco_category", "analyzer"))

In [ ]:
eligible_all = pd.concat([context_eligible, analyzer_eligible], ignore_index=True)
if eligible_all.empty:
    print("(no eligible comparison pairs to chart)")
else:
    comparison_chart = (eligible_all.pivot_table(index="comparison", columns="change", values="finding_key",
                                                 aggfunc="count", fill_value=0, observed=True)
                        .reindex(columns=[c for c in CHANGE_ORDER if c in set(eligible_all["change"])]).astype(int))
    change_colors = dict(zip(CHANGE_ORDER, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]))
    grouped_bar_chart(comparison_chart, "Finding-level changes between configurations (primary analysis: complete scans)",
                      "number of findings", colors=change_colors, save_as="configuration_comparisons.png")

all_pairs = pd.concat([context_pairs, analyzer_pairs], ignore_index=True)
all_diff = pd.concat([context_diff, analyzer_diff], ignore_index=True)
save_csv(context_diff, "context_comparison.csv")
save_csv(analyzer_diff, "analyzer_comparison.csv")

## 14 — Analysis D: how reliable are these results? And E: the manual-review file

Before any number above goes into a paper, this section answers: how many scans finished cleanly, which ones
Cisco reported as partially failed, how many folders could not be rebuilt, where the scanner verdict and the
finding severities tell different stories, and how trustworthy the reported locations are.

* **D1** — planned configurations, executed scans and outcome per configuration (complete / contract_only / partial /
  failed for executed scans; excluded_incomplete_folder and missing for configurations Cisco never scanned).
* **D2** — analyzer failures Cisco reported, one row per failure.
* **D3** — folder completeness, missing supporting files, export integrity.
* **D4** — scanner verdict vs finding severity per scan: `is_safe` only means "no CRITICAL/HIGH finding".
* **D5** — evidence check results per rule (Section 9).
* **D6** — comparison pairs and their eligibility.

**E — manual review.** `manual_review_template.csv` has one row per distinct security finding location (skill,
rule, file, reported line) across all complete, contract-only and partial scans. It shows Cisco's original rule, severity,
reported file and line, the redacted evidence, the evidence check, and the configurations where it appeared.
The blank columns are for people: `reviewer`, `evidence_verified` (did the reviewer find the evidence in the file?),
`legitimate_purpose` (e.g. "credential manager reads secrets by design"), `interpretation` (suggested values:
observed operation / scanner warning only / potential vulnerability / confirmed malicious / false positive), and
`notes`. **The notebook never fills these in**; manual verdicts stay in that file and are never merged back into
Cisco's output.

In [ ]:
failure_rows = []
for run in scan_runs:
    for failure in json.loads(run["analyzers_failed"]) if run["analyzers_failed"] else []:
        failure_rows.append({"skill_id": run["skill_id"], "configuration": run["configuration"], "status": run["status"],
                             "analyzer": failure["analyzer"], "failure_type": failure["type"], "error": failure["error"]})
failures_df = pd.DataFrame(failure_rows, columns=["skill_id", "configuration", "status", "analyzer", "failure_type", "error"])
# (rule ids for contract failures are kept in scan_summary.csv -> analyzers_failed)

if scan_runs_df.empty:
    print("No runs to assess.")
else:
    print("D1  Scan outcome per configuration")
    d1 = scan_runs_df.pivot_table(index="configuration", columns="outcome", values="skill_id", aggfunc="count", fill_value=0)
    d1 = d1.reindex(index=[c for c in CONFIG_ORDER + ["(export)"] if c in set(scan_runs_df["configuration"])],
                    columns=OUTCOME_ORDER, fill_value=0)
    d1.insert(0, "executed", scan_runs_df.groupby("configuration")["scan_executed"].sum().reindex(d1.index).astype(int))
    d1.insert(0, "planned", scan_runs_df.groupby("configuration").size().reindex(d1.index).astype(int))
    d1.loc["total"] = d1.sum()
    display(d1)

    print(f"\nD2  Analyzer failures reported by Cisco: {len(failures_df)}")
    if not failures_df.empty:
        display(failures_df)

    print("\nD3  Exported skills: folder completeness, missing supporting files, integrity")
    display(exports_df[["skill_id", "origin", "integrity", "supporting_files_exported", "files_missing", "listing_complete",
                        "folder_complete", "frontmatter_valid"]])
    print(f"    complete folders: {int((exports_df['folder_complete'] == True).sum())} / {len(exports_df)}"
          f"    invalid frontmatter: {int((exports_df['frontmatter_valid'] == 0).sum())}    export errors: {len(export_errors)}")

In [ ]:
SEVERITY_RANK = {name: rank for rank, name in enumerate(reversed(SEVERITY_ORDER))}   # INFO=0 ... CRITICAL=4


def highest(severities):
    values = [str(s) for s in severities if pd.notna(s)]
    return max(values, key=SEVERITY_RANK.get) if values else "none"


verdict_rows = []
for run in scan_runs:
    if (run["skill_id"], run["configuration"]) not in reports:
        continue
    these = findings_df[(findings_df["skill_id"] == run["skill_id"]) & (findings_df["configuration"] == run["configuration"])]
    sec = these[these["finding_kind"] == "security"]
    recomputed = highest(these["severity"])
    note = []
    if (recomputed if recomputed != "none" else "SAFE") != run["max_severity"]:
        note.append("Cisco max_severity differs from the severities of its own findings")
    if run["is_safe"] and len(sec):
        note.append(f"is_safe=True although there are {len(sec)} security finding(s) up to {highest(sec['severity'])}")
    if not run["is_safe"]:
        note.append(f"is_safe=False because of {int(sec['severity'].isin(['CRITICAL', 'HIGH']).sum())} CRITICAL/HIGH security finding(s)")
    if len(these) and not len(sec):
        note.append("only metadata/coverage findings")
    verdict_rows.append({"skill_id": run["skill_id"], "configuration": run["configuration"], "outcome": run["outcome"],
                         "is_safe": run["is_safe"], "cisco_max_severity": run["max_severity"],
                         "highest_security_severity": highest(sec["severity"]), "security_findings": len(sec),
                         "security_medium_or_lower": int(sec["severity"].isin(["MEDIUM", "LOW", "INFO"]).sum()),
                         "note": "; ".join(note) or "no findings"})
verdict_df = pd.DataFrame(verdict_rows)
print("D4  Scanner verdict vs finding severity")
display(verdict_df if not verdict_df.empty else pd.DataFrame({"note": ["no scans with a report"]}))

if not findings_df.empty:
    print("\nD5  Evidence check per rule (all findings in complete and partial scans; reported lines are never changed)")
    d5 = findings_df.pivot_table(index="rule_id", columns="evidence_status", values="finding_key", aggfunc="count", fill_value=0)
    d5 = d5.reindex(columns=[c for c in ["exact", "ambiguous", "mismatch", "unavailable"] if c in d5.columns])
    display(d5.sort_values(list(d5.columns), ascending=False))
    off = findings_df[findings_df["evidence_status"].isin(["mismatch", "ambiguous"])]
    if not off.empty:
        print("    Findings whose evidence is not on the reported line (first 15):")
        display(off[["skill_id", "configuration", "rule_id", "file_path", "reported_line_number", "evidence_status",
                     "evidence_candidate_lines", "evidence_detail"]].head(15))

print("\nD6  Comparison pairs and eligibility")
display(all_pairs if not all_pairs.empty else pd.DataFrame({"note": ["no pairs"]}))

In [ ]:
REVIEW_COLUMNS = ["review_id", "skill_id", "repo", "skill_path", "rule_id", "severity_reported", "cisco_category",
                  "research_category", "analyzer", "reported_file_path", "file_path", "reported_line_number",
                  "evidence_snippet", "evidence_status", "evidence_candidate_lines", "title",
                  "configurations", "run_statuses", "only_in_non_complete_scans", "cisco_finding_ids",
                  "reviewer", "evidence_verified", "legitimate_purpose", "interpretation", "notes"]

review_source = findings_df[findings_df["finding_kind"] == "security"]
review_rows = []
for (skill_id, location_key), group in review_source.groupby(["skill_id", "location_key"], sort=True):
    first = group.sort_values("severity").iloc[0]
    review_rows.append({
        "review_id": hashlib.sha256(f"{skill_id}|{location_key}".encode()).hexdigest()[:10],
        "skill_id": skill_id, "repo": first["repo"], "skill_path": first["skill_path"], "rule_id": first["rule_id"],
        "severity_reported": "; ".join(sorted({str(s) for s in group["severity"]}, key=SEVERITY_RANK.get, reverse=True)),
        "cisco_category": first["cisco_category"], "research_category": first["research_category"],
        "analyzer": "; ".join(sorted(set(group["analyzer"].dropna()))),
        "reported_file_path": first["reported_file_path"], "file_path": first["file_path"],
        "reported_line_number": first["reported_line_number"], "evidence_snippet": first["snippet"],
        "evidence_status": first["evidence_status"], "evidence_candidate_lines": first["evidence_candidate_lines"],
        "title": first["title"],
        "configurations": "; ".join(sorted(set(map(str, group["configuration"])), key=CONFIG_ORDER.index)),
        "run_statuses": "; ".join(sorted(set(group["run_status"]))),
        "only_in_non_complete_scans": not bool((group["run_outcome"] == "complete").any()),
        "cisco_finding_ids": "; ".join(sorted(set(group["finding_id"].dropna()))),
        "reviewer": "", "evidence_verified": "", "legitimate_purpose": "", "interpretation": "", "notes": "",
    })
review_df = pd.DataFrame(review_rows, columns=REVIEW_COLUMNS)
review_df["reported_line_number"] = review_df["reported_line_number"].astype("Int64")
print(f"E   {len(review_df)} distinct security finding location(s) for manual review "
      f"({int(review_df['only_in_non_complete_scans'].sum())} seen only in contract-only or partial scans).")
save_csv(review_df, "manual_review_template.csv")

## 15 — Write `results_summary.md`

This section writes a short, research-oriented summary of the run to `results_summary.md` in the run folder and
displays it below. **Every number in it is computed from the tables above** (`scan_runs_df`, `findings_df`,
the comparison tables, `review_df`); nothing is typed in by hand, so the same cell works for a one-skill pilot,
a batch, or a reanalysis. The limitation bullets are also generated: a bullet about contract-only or partial scans,
missing files or line-number mismatches only appears when the data contains such cases. Section C of the summary
shows the primary comparisons, the sensitivity analysis with contract-only scans, the sample size of each, and a
generated sentence per comparison saying whether including contract-only scans changes the result.

`md_table` is a small helper that turns a DataFrame into a Markdown table (pandas' own `to_markdown` needs an extra
package). The run's `run_metadata.json` is rewritten at the end with the list of output files.

In [ ]:
def md_cell(value):
    if isinstance(value, (list, dict)):
        value = json.dumps(value)
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return ""
    return str(value).replace("|", "\\|").replace("\n", " ")


def md_table(df, max_rows=20):
    """DataFrame -> Markdown table text (index included when it carries names)."""
    if df is None or df.empty:
        return "_No data._\n"
    table = df.reset_index() if any(name is not None for name in df.index.names) else df
    headers = [md_cell(" / ".join(map(str, c)) if isinstance(c, tuple) else str(c)) for c in table.columns]
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(md_cell(v) for v in row) + " |" for row in table.head(max_rows).itertuples(index=False)]
    if len(table) > max_rows:
        lines.append(f"\n_{len(table) - max_rows} more row(s) in the CSV files._")
    return "\n".join(lines) + "\n"


def describe_values(values):
    values = sorted({str(v) for v in values if pd.notna(v)})
    return ", ".join(values) if values else "not recorded"


def overview_table():
    COUNTS = run_counts()
    timestamps = scan_runs_df["report_timestamp"].dropna().sort_values()
    with_reports = findings_df["run_outcome"].value_counts()
    distinct_locations = findings_df.groupby(["skill_id", "location_key"]).ngroups if not findings_df.empty else 0
    distinct_security = security.groupby(["skill_id", "location_key"]).ngroups if not security.empty else 0
    dataset = (f"{DB_PATH.name} (SHA-256 {DB_SHA256[:12]}...)" if DB_SHA256 else
               f"{DB_PATH.name} (hash not computed)" if conn is not None else
               f"{(SOURCE_METADATA.get('database') or {}).get('path', 'unknown')} (from the saved run; database not available here)")
    rows = [
        ("Run ID", RUN_ID_USED),
        ("Run mode", "scan (Cisco executed in this run)" if RUN_MODE == "scan"
                     else f"reanalysis of saved run `{show_path(SOURCE_DIR)}` (Cisco not executed)"),
        ("Dataset", dataset),
        *([("Sample", f"{SAMPLE_INFO['sample_id']} ({SAMPLE_INFO['skills']} skills; manifest {SAMPLE_INFO['manifest']})")]
          if SAMPLE_INFO is not None else []),
        ("Scanner version that produced the reports", describe_values(scan_runs_df["scanner_version"])),
        ("Scan policy (name / preset)", f"{describe_values(scan_runs_df['policy_name'])} / {describe_values(scan_runs_df['policy_preset'])}"),
        ("Policy fingerprint(s)", ", ".join(f[:12] + "..." for f in scan_runs_df["policy_fingerprint"].dropna().unique()) or "not recorded"),
        ("Scans executed (report timestamps, UTC)", f"{str(timestamps.iloc[0])[:19].replace('T', ' ')} to "
                                                    f"{str(timestamps.iloc[-1])[:19].replace('T', ' ')}" if len(timestamps) else "no reports"),
        ("Analysis written", now_utc()),
        ("Skills selected / exported", f"{len(selected_df)} / {len(exports)}" + (f" ({len(export_errors)} export error(s))" if export_errors else "")),
        ("Skill folders complete / incomplete", f"{int((exports_df['folder_complete'] == True).sum())} / {int((exports_df['folder_complete'] != True).sum())}"),
        ("Configurations per skill", len(SCAN_CONFIGS)),
        ("Configurations planned (skills x configurations)", COUNTS["planned"]),
        ("Cisco scans executed", COUNTS["executed"]),
        ("Configurations excluded by design: incomplete GitSkills folder (full-folder scan not run)", COUNTS["excluded_incomplete_folder"]),
        *([("Configurations with no report (missing)", COUNTS["missing"])] if COUNTS["missing"] else []),
        *([("Configurations not run: export error", COUNTS["not_run_export_error"])] if COUNTS["not_run_export_error"] else []),
        ("Outcomes of the executed scans", ", ".join(f"{k}: {v}" for k, v in COUNTS["executed_outcomes"].items())),
        ("Finding records (all readable reports)", f"{len(findings_df)} (complete scans: {with_reports.get('complete', 0)}, "
                                                    f"contract-only scans: {with_reports.get('contract_only', 0)}, partial scans: {with_reports.get('partial', 0)})"),
        ("Security finding records in complete scans", len(security)),
        ("Distinct rule/location combinations (skill, rule, file, line), all findings", distinct_locations),
        ("Distinct rule/location combinations, security findings in complete scans", distinct_security),
    ]
    return pd.DataFrame(rows, columns=["item", "value"])

In [ ]:
def failure_list(value):
    """analyzers_failed is stored as JSON text; an empty value may be None or NaN (from a DataFrame)."""
    return json.loads(value) if isinstance(value, str) and value else []


def partial_reason(run):
    """Short label for why a scan is PARTIAL (the known contract issue alone never makes a scan PARTIAL)."""
    failures = failure_list(run["analyzers_failed"])
    labels = [f"Cisco analyzer failure: {f['type']}" for f in failures if "known Cisco issue" not in f["type"]]
    detail = str(run["status_detail"])
    for marker, label in (("rule-contract check failed", "Cisco rule-contract check failed"),
                          ("missing from analyzers_used", "requested analyzer did not run"),
                          ("LLM_ANALYSIS_FAILED", "LLM analyzer could not review the skill"),
                          ("scanner exit code", "non-zero scanner exit code")):
        if marker in detail:
            labels.append(label)
    return "; ".join(dict.fromkeys(labels)) or "other"


def contract_rules(outcome):
    """Rule IDs named in FindingContract failures of scans with the given outcome."""
    rules = set()
    for run in scan_runs:
        if run["outcome"] == outcome:
            rules |= {f["rule_id"] for f in failure_list(run["analyzers_failed"]) if f.get("rule_id")}
    return sorted(rules)


def limitation_bullets():
    bullets = []
    contract_runs = scan_runs_df[scan_runs_df["outcome"] == "contract_only"]
    if not contract_runs.empty:
        n_findings = int(findings_df["run_outcome"].eq("contract_only").sum())
        bullets.append(f"{len(contract_runs)} of {run_counts()['executed']} executed scan(s) were CONTRACT_ONLY: their only recorded problem is "
                       f"Cisco's rule-registration error for {', '.join(contract_rules('contract_only'))}. Their {n_findings} finding record(s) "
                       "are excluded from primary results and used only in the sensitivity analysis; they are not assumed complete or accurate.")
    partial_runs = scan_runs_df[scan_runs_df["outcome"] == "partial"]
    if not partial_runs.empty:
        reasons = partial_runs.apply(partial_reason, axis=1).value_counts()
        n_findings = int(findings_df["run_outcome"].eq("partial").sum())
        bullets.append(f"{len(partial_runs)} of {run_counts()['executed']} executed scan(s) were PARTIAL ("
                       + "; ".join(f"{label}: {n}" for label, n in reasons.items())
                       + f"). Their {n_findings} finding record(s) are preliminary evidence and are excluded from primary tables and comparisons.")
    counts = run_counts()
    failed = int(scan_runs_df["outcome"].eq("failed").sum())
    if failed:
        bullets.append(f"{failed} configuration(s) failed (no usable report); they produced no findings and are not counted as zero findings.")
    if counts["missing"]:
        bullets.append(f"{counts['missing']} configuration(s) have no saved report or run record; they are not counted as zero findings.")
    incomplete = exports_df[exports_df["folder_complete"] != True]
    if not incomplete.empty:
        excluded = counts["excluded_incomplete_folder"]
        bullets.append(f"{len(incomplete)} skill folder(s) are incomplete in GitSkills ({int(incomplete['files_missing'].sum())} supporting file(s) "
                       f"without stored text). This is a design exclusion, not a scanner failure: {excluded} of {counts['planned']} planned "
                       f"configuration(s) (their full-folder scans) were not run, so {counts['executed']} Cisco scans were executed. "
                       "Their SKILL.md-only scans are kept, but their context-comparison pairs are reported as `excluded: incomplete folder`, "
                       "and no finding count of zero is implied for the excluded configurations.")
    problems = exports_df[~exports_df["integrity"].astype(str).str.startswith("ok")]
    if not problems.empty:
        bullets.append(f"{len(problems)} copied skill folder(s) failed the integrity check: " + "; ".join(problems["integrity"]))
    off = findings_df[findings_df["evidence_status"].isin(["mismatch", "ambiguous"])]
    if not off.empty:
        rules = off["rule_id"].value_counts()
        bullets.append(f"{len(off)} of {len(findings_df)} finding record(s) do not have their evidence on the reported line "
                       f"(mismatch {int(off['evidence_status'].eq('mismatch').sum())}, ambiguous {int(off['evidence_status'].eq('ambiguous').sum())}); "
                       f"rules involved: {', '.join(f'{r} ({n})' for r, n in rules.items())}. Reported line numbers are kept unchanged.")
        if any(str(r).startswith("MDBLOCK_") for r in rules.index):
            bullets.append("For MDBLOCK_* rules Cisco 2.3.1 counts lines from the code-block fence and, in SKILL.md, from the end of the "
                           "YAML frontmatter, so their reported lines point above the code; use evidence_candidate_lines to locate it.")
    unavailable = int(findings_df["evidence_status"].eq("unavailable").sum())
    if unavailable:
        bullets.append(f"{unavailable} finding record(s) could not be checked against the file (no file, line or evidence text).")
    if len(selected_df) < 30:
        bullets.append(f"Only {len(selected_df)} skill(s) were analysed: these are pilot observations, not population estimates.")
    bullets += [
        "Scanner findings are rule matches on operations; they are not confirmed vulnerabilities, and nothing here is a false-negative "
        "claim (there is no ground truth).",
        "Finding counts are per matching line; one script can contribute many findings. Distinct skills are reported separately.",
        "Comparisons match findings by rule, file, reported line and evidence; equivalent behavior reported under different rule IDs "
        "is not matched automatically.",
    ]
    return bullets


if RUN_DIR is None:
    print("No run folder: results_summary.md is not written.")
else:
    sev_cols = ["CRITICAL", "HIGH", "MEDIUM", "LOW"] + (["INFO"] if security["severity"].eq("INFO").any() else [])
    severity_table = count_table(security, "configuration", "severity").reindex(index=PRIMARY_CONFIGS, columns=sev_cols, fill_value=0) \
        if not security.empty else pd.DataFrame()
    category_table = pd.DataFrame()
    if not security.empty:
        category_table = count_table(security, "cisco_category", "configuration")
        category_table["distinct_skills"] = security.groupby("cisco_category")["skill_id"].nunique()
    research_table = pd.DataFrame()
    if not security.empty:
        research_table = count_table(security, "research_category", "configuration")
        research_table = research_table.reindex(RESEARCH_CATEGORIES + [c for c in research_table.index if c not in RESEARCH_CATEGORIES], fill_value=0)
        research_table["distinct_skills"] = security.groupby("research_category")["skill_id"].nunique().reindex(research_table.index, fill_value=0)
    top_rules = (security.groupby(["rule_id", "cisco_category"], observed=True)
                 .agg(findings=("finding_key", "size"), distinct_skills=("skill_id", "nunique"),
                      severities=("severity", lambda s: ", ".join(sorted(set(map(str, s))))))
                 .sort_values("findings", ascending=False).head(10)) if not security.empty else pd.DataFrame()
    location_table = count_table(security, "location", "configuration")
    outcome_table = scan_runs_df.pivot_table(index="configuration", columns="outcome", values="skill_id", aggfunc="count", fill_value=0) \
        .reindex(index=[c for c in CONFIG_ORDER + ["(export)"] if c in set(scan_runs_df["configuration"])],
                 columns=OUTCOME_ORDER, fill_value=0) if not scan_runs_df.empty else pd.DataFrame()
    if not outcome_table.empty:
        outcome_table.insert(0, "executed", scan_runs_df.groupby("configuration")["scan_executed"].sum().reindex(outcome_table.index).astype(int))
        outcome_table.insert(0, "planned", scan_runs_df.groupby("configuration").size().reindex(outcome_table.index).astype(int))
        outcome_table.loc["total"] = outcome_table.sum()
    additional_files = context_eligible[context_eligible["change"] == "additional"].groupby(["location", "file_path"], dropna=False) \
        .agg(findings=("finding_key", "size"), skills=("skill_id", "nunique"), rules=("rule_id", lambda s: ", ".join(sorted(set(s))))) \
        if not context_eligible.empty else pd.DataFrame()
    non_primary_table = non_primary.groupby(["run_status", "configuration", "rule_id", "severity"], observed=True).size() \
        .rename("findings").to_frame() if not non_primary.empty else pd.DataFrame()
    invalid = findings_df[findings_df["run_outcome"].eq("contract_only") & findings_df["cisco_rule_contract"].eq("invalid")]
    evidence_table = findings_df.pivot_table(index="rule_id", columns="evidence_status", values="finding_key", aggfunc="count", fill_value=0) \
        if not findings_df.empty else pd.DataFrame()

    S = []
    S.append(f"# Cisco Skill Scanner results: `{RUN_ID_USED}`\n")
    S.append(f"Generated automatically by `notebooks/01_cisco_scanner_dataset_ready.ipynb` on {now_utc()}. Every number below is "
             "computed from the files in this run folder. A finding is a Cisco rule match on an operation, not a confirmed vulnerability.\n")
    S.append("## A. Experiment overview\n")
    S.append(md_table(overview_table()))
    S.append("\nPlanned configurations, executed Cisco scans and outcomes per configuration. complete = primary results; contract_only = "
             "only Cisco's known rule-registration error, used in the sensitivity analysis only; partial = other analyzer problems, "
             "preliminary only; failed = Cisco ran but produced no usable report; excluded_incomplete_folder = design exclusion, "
             "Cisco was not run because GitSkills did not store the whole folder; missing = no saved report:\n")
    S.append(md_table(outcome_table))
    if exports_df["sample_stratum"].notna().any():
        stratum_rows = []
        for stratum, group in exports_df.groupby("sample_stratum", sort=False):
            runs = scan_runs_df[scan_runs_df["sample_stratum"] == stratum]
            sec = security[security["sample_stratum"] == stratum]
            row = {"stratum": stratum, "skills": len(group), "complete folders": int((group["folder_complete"] == True).sum())}
            row.update({"configurations planned": len(runs), "scans executed": int(runs["scan_executed"].sum())})
            row.update({f"{'configurations' if o in ('excluded_incomplete_folder', 'missing') else 'scans'} {o}": int((runs["outcome"] == o).sum())
                        for o in OUTCOME_ORDER})
            row.update({"skills with >= 1 security finding (complete scans)": int(sec["skill_id"].nunique()),
                        "security finding records (complete scans)": len(sec)})
            stratum_rows.append(row)
        S.append("\nResults by sampling stratum. Strata were sampled with equal allocation, so counts within a stratum can be "
                 "reported directly, but proportions for the whole population need the design weights in "
                 "`sample/sampling_metadata.json`:\n")
        S.append(md_table(pd.DataFrame(stratum_rows)))
    S.append("\n## B. Security findings (complete scans only)\n")
    S.append("B1. Findings by severity\n")
    S.append(md_table(severity_table))
    S.append("\nB2. Findings by Cisco category (per configuration) and distinct skills affected (any complete configuration)\n")
    S.append(md_table(category_table))
    S.append("\nB3. Findings by research category (our mapping; categories without Cisco rules stay at zero)\n")
    S.append(md_table(research_table))
    S.append("\nB4. Most frequently triggered security rules\n")
    S.append(md_table(top_rules, max_rows=10))
    S.append("\nB5. Where the findings point\n")
    S.append(md_table(location_table))
    S.append("\nB6. Findings outside the primary group: contract-only and partial scans (excluded from B1-B5)\n")
    S.append(md_table(non_primary_table))
    S.append("\n## C. Configuration comparisons\n")
    S.append("The **primary analysis** uses only pairs in which both scans are COMPLETE (and, for SKILL.md-only vs full-folder, the "
             "folder is complete). The **sensitivity analysis** adds pairs that contain a CONTRACT_ONLY scan. Contract-only scans are "
             "not assumed complete or accurate; the sensitivity analysis only shows whether including them would change a result.\n")
    S.append("C1. Sample sizes and excluded pairs per comparison (`excluded_*` = pairs left out of the primary analysis, by reason; "
             "contract-only pairs are added back in the sensitivity analysis)\n")
    S.append(md_table(sample_sizes(all_pairs)))
    S.append("\nC2. Primary analysis (complete scans only): finding-level changes summed over the pairs used\n")
    S.append(md_table(comparison_totals(all_diff, all_pairs, PRIMARY_PAIR_STATUSES)))
    S.append("\nC3. Files that contribute the additional findings in the primary SKILL.md-only vs full-folder comparison "
             "(a SKILL.md row means Cisco re-read the instructions once the scripts were present)\n")
    S.append(md_table(additional_files))
    S.append("\nC4. Sensitivity analysis (complete + contract-only scans): finding-level changes summed over the pairs used\n")
    S.append(md_table(comparison_totals(all_diff, all_pairs, SENSITIVITY_PAIR_STATUSES)))
    S.append("\nC5. Does including contract-only scans change the comparisons?\n")
    S += [f"- {line}" for line in sensitivity_effect(all_diff, all_pairs)] or ["- No comparison pairs."]
    if not invalid.empty:
        S.append(f"- {len(invalid)} finding record(s) in contract-only scans carry Cisco's own `rule_contract = invalid` flag "
                 f"(rules: {', '.join(sorted(set(invalid['rule_id'])))}); Cisco excludes these from its policy gating.")
    S.append("\nC6. Pair status per skill\n")
    S.append(md_table(all_pairs.drop(columns=["skill_id"]) if len(exports) == 1 and not all_pairs.empty else all_pairs))
    S.append("\n## D. Scanner reliability\n")
    S.append("D1. Analyzer failures reported by Cisco\n")
    S.append(md_table(failures_df))
    S.append("\nD2. Skill folders\n")
    S.append(md_table(exports_df[["skill_id", "supporting_files_exported", "files_missing", "folder_complete", "integrity"]]))
    S.append("\nD3. Scanner verdict vs finding severity\n")
    S.append(md_table(verdict_df.drop(columns=["skill_id"]) if len(exports) == 1 and not verdict_df.empty else verdict_df))
    S.append("\nD4. Evidence check of reported locations (counts of finding records per rule)\n")
    S.append(md_table(evidence_table))
    S.append("\n## E. Manual security review\n")
    S.append(f"`manual_review_template.csv` lists {len(review_df)} distinct security finding location(s) "
             f"({int(review_df['only_in_non_complete_scans'].sum())} seen only in contract-only or partial scans) with Cisco's rule, severity, reported file and line, "
             "redacted evidence and evidence check. The columns `reviewer`, `evidence_verified`, `legitimate_purpose`, `interpretation` "
             "and `notes` are intentionally blank; no finding has been labelled as a vulnerability or a false positive.\n")
    if SAVED_FIGURES:
        S.append("\n## Figures\n")
        S += [f"![{title}](figures/{path.name})\n" for path, title in SAVED_FIGURES]
    S.append("\n## Limitations\n")
    S += [f"- {b}" for b in limitation_bullets()]
    S.append("\n\n## Files in this run folder\n")
    S += [f"- `{name}`: {what}" for name, what in [
        ("scan_summary.csv", "one row per skill x configuration: status, outcome, analyzer failures, verdict, counts, versions"),
        ("individual_findings.csv", "one row per Cisco finding: original fields, normalized path, evidence check"),
        ("context_comparison.csv / analyzer_comparison.csv", "finding-level comparisons; pair_status says primary (eligible), sensitivity or preliminary"),
        ("manual_review_template.csv", "blank reviewer columns for manual interpretation"),
        ("run_metadata.json", "settings, dataset hash, scanner versions, policy fingerprints, outcome counts"),
        ("<skill_id>/reports/*.json", "raw Cisco reports, unmodified"),
        ("<skill_id>/manifest.json", "exported files with SHA-256 hashes and missing files"),
        ("figures/*.png", "the charts above")]]
    summary_text = "\n".join(S) + "\n"
    (RUN_DIR / "results_summary.md").write_text(summary_text, encoding="utf-8")
    print(f"saved {show_path(RUN_DIR / 'results_summary.md')}")
    save_run_metadata("complete")
    display(Markdown(summary_text.replace("](figures/", f"]({(RUN_DIR / 'figures').as_uri()}/")))

## 16 — Interpretation, limitations, and what comes next

### Reading the results

* **A finding is an observed operation plus a rule match.** A network rule firing on `requests.post(...)` in a
  webhook-alerting skill is correct detection of a legitimate operation. Whether it is a vulnerability is a manual
  judgement recorded in `manual_review_template.csv`, never in Cisco's columns.
* **`is_safe` is coarse.** It only says "no CRITICAL/HIGH". Use the severity and category tables (D4 shows where
  they disagree).
* **Contract-only scans are a sensitivity check, not results.** Their only recorded problem is Cisco's known
  rule-registration error, but Cisco still flagged its own output as inconsistent. Report primary numbers from
  complete scans, and use the sensitivity analysis (summary section C5) to say whether the conclusion would change.
  If a comparison has `primary_n = 0`, it cannot be answered from complete scans alone; say so.
* **Partial scans are not failures and not results.** Cisco produced findings, but also told us an analyzer did
  not finish cleanly for another reason. Their findings are kept (A8, B5, `individual_findings.csv` with
  `run_status = PARTIAL`) and reported as preliminary evidence.
* **Reported line numbers are Cisco's.** The evidence check (D5) shows how often they point at the evidence. For
  the paper, cite locations that are `exact` or that a reviewer verified.
* **Never call something a false negative from this notebook alone.** There is no ground truth here.
* **Counts are per finding; skills are counted separately.** One script with eight `requests.get` calls produces
  eight findings.

### What Cisco (2.3.1, `balanced`) covers for RQ1-B

Well covered by deterministic rules: hardcoded secrets and credential-file reads; outbound network calls and
exfiltration-shaped POSTs/sockets; eval/exec, `os.system`, shell pipelines; prompt-injection phrases and Unicode
steganography; undeclared network use versus the manifest; unpinned/untrusted package installs; obfuscation and
embedded binaries. **Not covered as such:** general file-system modification (only path traversal and hidden-file
globbing), and account modification or other sensitive external actions. The LLM judge is the only Cisco component
that could reason about *purpose*; it is optional, paid, and sends skill text to a provider.

### Known limitations of this setup

* **Context is bounded by GitSkills.** Folders with binaries, oversized files or more than the capture limit are
  incomplete; their full-folder scans are excluded by design (not run) and reported as `excluded_incomplete_folder`.
* **Only representatives are scanned.** Copies (`dedup_primary = 0`) resolve to their representative.
* **Frontmatter.** Skills with invalid frontmatter are scanned with Cisco's fallback (`SKILL_LOAD_FALLBACK_USED`).
* **Scanner version and policy are fixed per run** and recorded; mixing versions or policies invalidates
  finding-level comparisons.
* **Runtime.** Each scanner call takes about 5–10 s of wall time. Four configurations per skill ≈ 30 s per skill.

### Scaling to the 400-skill stratified sample

1. **Sampling, not keyword search.** `03_stratified_sample.ipynb` draws a seeded stratified sample of representatives
   and saves a sample manifest with population sizes and design weights; scan it with `SELECTION_MODE = "manifest"`.
2. **One run folder per batch.** The run-ID mechanism already isolates batches; add a study-level folder that lists
   its batch run IDs.
3. **Resumability.** Run records (`*.run.json`) already make each scan's outcome readable from disk; add "skip
   configurations that already have a run record" for an interrupted batch.
4. **Throughput.** 400 skills × 4 configurations ≈ 1,600 scanner calls ≈ 2.5–4.5 h serially.
5. **Aggregation.** Point Sections 9–15 at several run folders (reanalysis of many sources) with skill-level
   denominators per stratum.
6. **Manual review protocol.** Two reviewers, the `interpretation` vocabulary above, and inter-rater agreement on a
   subsample, before any claim about legitimate vs problematic behavior.

## 17 — Build the researcher report

This section turns the run folder into a researcher-facing report in `results/reports/<database>/<run ID>/`:

| folder | what is in it |
|---|---|
| `01_IMPORTANT_RESULTS/` | `START_HERE.md`, `dashboard.html` (self-contained, opens offline), coverage / severity / comparison tables, a copy of `results_summary.md` |
| `02_DETAILED_ANALYSIS/` | one page per skill with security alerts, rule and analyzer breakdowns, the full finding tables, figures |
| `03_RAW_SCANNER_OUTPUT/` | an index of every raw Cisco report with its SHA-256 (the reports themselves are **not** copied), metadata copies |
| `04_HUMAN_VALIDATION/` | validation status and snapshots of the manual-review worksheet, when one exists for this run |

The run folder is only read. Every text output is redacted (secret-like strings become `[REDACTED]`), absolute
paths become repository-relative, and the finished report is audited for both before it is published. Unlike
the run folder, `results/reports/` is meant to be committed. The code lives in `src/gitskills_reports/`;
`notebooks/05_build_reports.ipynb` rebuilds any or all reports later. Set `BUILD_REPORT = False` to skip this step.

In [ ]:
BUILD_REPORT = True        # False = skip; build it later with notebooks/05_build_reports.ipynb

if not BUILD_REPORT or RUN_DIR is None:
    print("Report not built:", "BUILD_REPORT is False" if not BUILD_REPORT else "no run folder was created")
else:
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from gitskills_reports import build_report, load_run, refresh_index

    REPORT_DIR = build_report(load_run(RUN_DIR), ROOT)
    refresh_index(ROOT)
    print("Report     :", show_path(REPORT_DIR))
    print("Start here :", show_path(REPORT_DIR / "01_IMPORTANT_RESULTS" / "START_HERE.md"))
    print("Dashboard  :", (REPORT_DIR / "01_IMPORTANT_RESULTS" / "dashboard.html").as_uri())
    print("All reports:", show_path(ROOT / "results" / "reports" / "index.html"))